<div style="direction: rtl; white-space: normal; line-height: 1;">

# Notebook 02 — Website Acceptance

## پذیرش و اعتبارسنجی خروجی Website Ingestion

Notebook `01_website_ingestion.ipynb` قبلاً فرآیند استخراج محتوای Website را انجام داده و خروجی فعلی شامل Manifest با 529 رکورد است.

در این Notebook هیچ Crawl یا Ingestion جدیدی انجام نمی‌شود.

هدف این مرحله این است که بررسی کنیم آیا خروجی فعلی Website از نظر ساختار داده، هویت اسناد، Lineage، Duplicate، Attribution، Citation Locator و سایر معیارهای کیفیت، شرایط لازم برای ورود به Knowledge Base فاز دوم را دارد یا خیر.

### اصل مهم

وجود 529 رکورد در Manifest به معنی پذیرفته‌شدن 529 مقاله نیست.

هر رکورد باید بر اساس شواهد واقعی بررسی شود و نتیجه آن در یکی از وضعیت‌های مناسب Acceptance قرار گیرد.

### حوزه بررسی این Notebook

در این Notebook موارد زیر بررسی خواهند شد:

- صحت و ثبات Artifact ورودی
- ساختار و Schema واقعی Manifest
- Completeness فیلدهای ضروری
- Stable ID
- URL و Source Identity
- Raw Artifact و Hash
- Lineage
- Duplicate
- Citation Locator
- Attribution و شواهد نویسندگی / نظارت
- کیفیت متن
- وضعیت رکوردهای ناقص یا مشکوک
- گزارش نهایی Acceptance
- PASS / FAIL Gate

### سیاست Attribution

برچسب‌هایی مانند `primary_author` یا `supervised` به‌تنهایی اثبات نویسندگی نیستند.

برای Attribution باید Evidence واقعی مانند Byline، نام نویسنده، اطلاعات صفحه یا سایر شواهد معتبر وجود داشته باشد.

در ادامه باید بتوانیم میان این مفاهیم تفکیک ایجاد کنیم:

- نویسنده مستقیم
- نویسنده دیگر تحت نظارت تحریریه
- متخصص یا مرجع علمی
- ناظر یا ویراستار
- وضعیت نامشخص

### ورودی اصلی

`data/phase2/manifests/website_manifest_v1.json`

### خروجی مورد انتظار

در پایان این Notebook باید یک گزارش Acceptance قابل اندازه‌گیری داشته باشیم که دقیقاً مشخص کند:

- چه تعداد رکورد بررسی شده‌اند.
- چه مشکلاتی وجود دارد.
- چه رکوردهایی قابل پذیرش هستند.
- چه رکوردهایی نیاز به Repair یا Quarantine دارند.
- آیا Dataset سایت مجاز به عبور به مراحل بعدی فاز دوم هست یا خیر.

</div>

<div style="direction: rtl; white-space: normal; line-height: 1;">

### شناسایی Root پروژه و Artifact ورودی Website

اولین کار این Notebook این است که بدون استفاده از مسیر Hard-code شده، Root واقعی Repository را پیدا کند و فایل Manifest ورودی Website را شناسایی کند.

این مرحله عمداً هیچ داده‌ای را تغییر نمی‌دهد.

همچنین SHA-256 فایل ورودی محاسبه می‌شود تا هویت Artifact مورد بررسی دقیقاً مشخص باشد.

بر اساس Baseline ثبت‌شده در Notebook 00، انتظار داریم Manifest فعلی دارای مشخصات زیر باشد:

- فایل: `website_manifest_v1.json`
- تعداد رکورد: `529`
- SHA-256: `35b41bfd6d2bbdc6e5799f55a864ce1f64cb798208bf0b55244c72a6cbda5be0`

این مقادیر در این Notebook صرفاً اعتماد نمی‌شوند؛ دوباره از روی فایل واقعی محاسبه خواهند شد.

**معیار PASS:**

- Root پروژه شناسایی شود.
- Manifest وجود داشته باشد.
- JSON قابل خواندن باشد.
- ساختار سطح اصلی `list` باشد.
- تعداد رکوردها 529 باشد.
- Hash فایل با Baseline ثبت‌شده یکسان باشد.

هرگونه تفاوت در Hash یا تعداد رکوردها به این معنی است که ورودی Website از زمان ثبت Baseline تغییر کرده و Acceptance نباید بدون بررسی علت ادامه پیدا کند.

</div>

In [1]:
# Code Cell 01

from pathlib import Path
import hashlib
import json

current_working_directory = Path.cwd().resolve()
search_locations = [
    current_working_directory,
    *current_working_directory.parents,
]

project_root = None

for location in search_locations:
    required_paths = [
        location / ".git",
        location / "notebooks",
        location / "data",
    ]

    if all(path.exists() for path in required_paths):
        project_root = location
        break

if project_root is None:
    raise RuntimeError(
        "Project root could not be identified from the current working directory."
    )

website_manifest_path = (
    project_root
    / "data"
    / "phase2"
    / "manifests"
    / "website_manifest_v1.json"
)

assert website_manifest_path.is_file(), (
    f"Website manifest not found: {website_manifest_path}"
)

manifest_bytes = website_manifest_path.read_bytes()
manifest_sha256 = hashlib.sha256(manifest_bytes).hexdigest()

with website_manifest_path.open("r", encoding="utf-8") as file:
    website_manifest = json.load(file)

expected_manifest_sha256 = (
    "35b41bfd6d2bbdc6e5799f55a864ce1f64cb798208bf0b55244c72a6cbda5be0"
)
expected_record_count = 529

assert isinstance(website_manifest, list), (
    f"Unexpected manifest type: {type(website_manifest).__name__}"
)

assert len(website_manifest) == expected_record_count, (
    f"Record count mismatch: expected {expected_record_count}, "
    f"found {len(website_manifest)}"
)

assert manifest_sha256 == expected_manifest_sha256, (
    "Manifest SHA-256 does not match the Phase 2 baseline."
)

print("Project root:", project_root)
print("Manifest:", website_manifest_path.relative_to(project_root))
print("Manifest SHA-256:", manifest_sha256)
print("Top-level type:", type(website_manifest).__name__)
print("Record count:", len(website_manifest))

print("\nWEBSITE INPUT START GATE: PASS")

Project root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Manifest: data/phase2/manifests/website_manifest_v1.json
Manifest SHA-256: 35b41bfd6d2bbdc6e5799f55a864ce1f64cb798208bf0b55244c72a6cbda5be0
Top-level type: list
Record count: 529

WEBSITE INPUT START GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

## سیاست Attribution و درجه اعتبار مقالات Website

تمام 529 مقاله استخراج‌شده از Website در یک سطح Attribution قرار نمی‌گیرند.

بر اساس ساختار محتوایی این Website، باید میان «نویسندگی مستقیم» و «نظارت و تأیید تحریری» تفاوت قائل شویم.

### درجه ۱ — Direct Author

این گروه شامل مقالاتی است که مستقیماً به قلم دکتر کمیل رودی منتشر شده‌اند.

این منابع بالاترین سطح Attribution را برای پاسخ‌هایی دارند که نیاز دارند دیدگاه یا بیان مستقیم ایشان مشخص شود.

بر اساس اطلاعات فعلی پروژه، انتظار می‌رود حدود 76 مقاله در این گروه قرار داشته باشند.

### درجه ۲ — Editorially Supervised / Approved

این گروه شامل مقالاتی است که لزوماً نویسنده مستقیم آن‌ها دکتر کمیل رودی نیست، اما تحت سردبیری، نظارت و تأیید ایشان منتشر شده‌اند.

این مقالات همچنان بخشی از Knowledge Corpus مورد قبول پروژه محسوب می‌شوند و نباید به‌عنوان منابع نامرتبط یا خارج از حوزه دانش ایشان حذف شوند.

بر اساس اطلاعات فعلی پروژه، انتظار می‌رود حدود 453 مقاله در این گروه قرار داشته باشند.

### تفاوت کاربرد دو درجه

هر دو گروه می‌توانند در Retrieval و پاسخ‌گویی عمومی سیستم مورد استفاده قرار گیرند، اما سیستم نباید این دو نوع Attribution را با یکدیگر مخلوط کند.

برای مثال، در پاسخ به پرسشی مانند:

`دیدگاه مجموعه آموزشی دکتر رودی درباره پس‌انداز چیست؟`

استفاده از هر دو درجه می‌تواند مجاز باشد.

اما در پرسشی مانند:

`خود دکتر کمیل رودی درباره پس‌انداز چه گفته است؟`

منابع دارای نویسندگی مستقیم باید مبنای اصلی پاسخ باشند و مقاله‌ای که صرفاً تحت نظارت ایشان منتشر شده نباید به‌عنوان نقل یا نوشته مستقیم ایشان معرفی شود.

### اصل اجرایی

بنابراین Website Corpus به‌صورت کلی بخشی از Knowledge Base مورد تأیید پروژه است، اما Attribution هر رکورد باید به‌صورت مستقل حفظ شود.

در ادامه این Notebook تلاش می‌کنیم وجود شواهد و Metadata لازم برای تشخیص این دو سطح را بررسی کنیم و بدون شواهد کافی، نوع نویسندگی مستقیم یک رکورد را حدس نزنیم.

</div>

<div style="direction: rtl; white-space: normal; line-height: 1;">

### کشف Schema واقعی رکوردهای Website

قبل از تعریف Validationهای Attribution، Lineage و Quality باید ساختار واقعی 529 رکورد Manifest را بررسی کنیم.

در این مرحله هیچ Schema از پیش فرض نمی‌شود.

تمام نام فیلدهایی که در رکوردهای Manifest وجود دارند استخراج می‌شوند و مشخص می‌شود:

- چه فیلدهایی در همه رکوردها حضور دارند.
- چه فیلدهایی فقط در بخشی از رکوردها وجود دارند.
- نوع داده هر فیلد چیست.
- نمونه اولین رکورد چه ساختاری دارد.

این بررسی به ما اجازه می‌دهد Validationهای بعدی را بر اساس داده واقعی بسازیم، نه بر اساس حدس یا ساختار مورد انتظار.

**معیار PASS:**

- هر 529 رکورد از نوع `dict` باشند.
- حداقل یک فیلد در Dataset وجود داشته باشد.
- Schema واقعی با موفقیت استخراج شود.

</div>

In [2]:
# Code Cell 02

from collections import Counter, defaultdict

assert all(isinstance(record, dict) for record in website_manifest), (
    "One or more manifest records are not JSON objects."
)

field_presence = Counter()
field_types = defaultdict(Counter)

for record in website_manifest:
    for field_name, value in record.items():
        field_presence[field_name] += 1
        field_types[field_name][type(value).__name__] += 1

all_fields = sorted(field_presence.keys())

print("Total records:", len(website_manifest))
print("Unique fields:", len(all_fields))

print("\nField coverage:")
for field_name in all_fields:
    coverage_count = field_presence[field_name]
    coverage_percent = coverage_count / len(website_manifest) * 100
    type_summary = ", ".join(
        f"{type_name}:{count}"
        for type_name, count in field_types[field_name].items()
    )

    print(
        f"{field_name:<30} "
        f"{coverage_count:>3}/{len(website_manifest)} "
        f"({coverage_percent:6.2f}%) "
        f"types=[{type_summary}]"
    )

print("\nFirst record keys:")
print(sorted(website_manifest[0].keys()))

print("\nWEBSITE SCHEMA DISCOVERY GATE: PASS")

Total records: 529
Unique fields: 10

Field coverage:
article_id                     529/529 (100.00%) types=[str:529]
author                         529/529 (100.00%) types=[NoneType:453, str:76]
canonical_url                  529/529 (100.00%) types=[str:529]
content_origin                 529/529 (100.00%) types=[str:529]
knowledge_owner                529/529 (100.00%) types=[str:529]
source_id                      529/529 (100.00%) types=[str:529]
source_type                    529/529 (100.00%) types=[str:529]
status                         529/529 (100.00%) types=[str:529]
url                            529/529 (100.00%) types=[str:529]
version                        529/529 (100.00%) types=[int:529]

First record keys:
['article_id', 'author', 'canonical_url', 'content_origin', 'knowledge_owner', 'source_id', 'source_type', 'status', 'url', 'version']

WEBSITE SCHEMA DISCOVERY GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### تفسیر Schema فعلی Website

تمام 529 رکورد Manifest دارای 10 فیلد یکسان هستند و از نظر حضور فیلدها در این سطح، Schema کاملاً یکنواخت است.

مهم‌ترین نتیجه این بررسی مربوط به فیلد `author` است.

در 76 رکورد، مقدار `author` به‌صورت متنی ثبت شده و در 453 رکورد مقدار این فیلد `None` است.

این توزیع عددی با تفکیک شناخته‌شده پروژه میان مقالات دارای نویسندگی مستقیم و سایر مقالات Website سازگار است.

با این حال، مقدار `None` در فیلد `author` به‌تنهایی اثبات نمی‌کند که رکورد مربوطه تحت سردبیری یا نظارت دکتر کمیل رودی بوده است.

طبق دانش و سیاست محتوایی پروژه، این 453 مقاله به‌عنوان محتوای درجه دوم تحت سردبیری، نظارت و تأیید ایشان در نظر گرفته می‌شوند؛ اما باید بررسی شود که آیا این رابطه در Metadata فعلی نیز به‌صورت صریح و قابل ردیابی ثبت شده است یا خیر.

در مراحل بعد نباید مفهوم `author=None` را با «نامرتبط بودن با دکتر رودی» یکسان تلقی کنیم.

</div>

<div style="direction: rtl; white-space: normal; line-height: 1;">

### بررسی توزیع Metadata و Attribution موجود

در این مرحله مقادیر واقعی فیلدهای اصلی Metadata بررسی می‌شوند تا مشخص شود Manifest فعلی درباره مالک دانش، منشأ محتوا، نوع منبع، وضعیت رکورد و نویسنده چه اطلاعاتی ثبت کرده است.

هدف اصلی این است که بفهمیم آیا رابطه مقالات درجه دوم با دکتر کمیل رودی در Metadata فعلی به‌صورت صریح وجود دارد یا صرفاً به‌عنوان دانش بیرونی پروژه شناخته شده است.

موارد زیر بررسی می‌شوند:

- توزیع مقادیر `author`
- توزیع `knowledge_owner`
- توزیع `content_origin`
- توزیع `source_type`
- توزیع `status`
- تعداد مقالات دارای نویسنده مستقیم
- تعداد مقالات بدون نویسنده مستقیم

در این مرحله هیچ Metadata جدیدی ساخته یا اصلاح نمی‌شود.

**معیار PASS:**  
توزیع تمام فیلدهای مورد بررسی بدون خطا استخراج شود و مجموع رکوردها در هر فیلد با 529 رکورد Manifest سازگار باشد.

</div>

In [3]:
# Code Cell 03

metadata_fields = [
    "author",
    "knowledge_owner",
    "content_origin",
    "source_type",
    "status",
]

metadata_distributions = {}

for field_name in metadata_fields:
    distribution = Counter(
        record.get(field_name)
        for record in website_manifest
    )

    metadata_distributions[field_name] = distribution

    print(f"\n{field_name}:")
    for value, count in distribution.most_common():
        display_value = "<NONE>" if value is None else repr(value)
        print(f"  {display_value}: {count}")

direct_author_count = sum(
    1
    for record in website_manifest
    if isinstance(record.get("author"), str)
    and record["author"].strip()
)

missing_direct_author_count = len(website_manifest) - direct_author_count

print("\nAttribution summary:")
print("Direct author populated:", direct_author_count)
print("Direct author missing:", missing_direct_author_count)
print("Total records:", len(website_manifest))

assert direct_author_count + missing_direct_author_count == len(website_manifest)

for field_name, distribution in metadata_distributions.items():
    assert sum(distribution.values()) == len(website_manifest), (
        f"Distribution count mismatch for field: {field_name}"
    )

print("\nWEBSITE METADATA DISTRIBUTION GATE: PASS")


author:
  <NONE>: 453
  'Dr. Komeil Roudi': 76

knowledge_owner:
  'Dr. Komeil Roudi': 529

content_origin:
  'supervised': 453
  'primary_author': 76

source_type:
  'website': 529

status:
  'discovered': 529

Attribution summary:
Direct author populated: 76
Direct author missing: 453
Total records: 529

WEBSITE METADATA DISTRIBUTION GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### تفسیر رسمی Attribution در Website Corpus

بررسی Metadata نشان داد ساختار فعلی Manifest، تفکیک مورد انتظار پروژه را به‌صورت صریح ثبت کرده است.

از مجموع 529 مقاله:

- 76 مقاله دارای `content_origin = primary_author` هستند و نام `Dr. Komeil Roudi` به‌عنوان `author` مستقیم آن‌ها ثبت شده است.
- 453 مقاله دارای `content_origin = supervised` هستند و `author` مستقیم برای آن‌ها ثبت نشده است.
- در تمام 529 رکورد، مقدار `knowledge_owner` برابر `Dr. Komeil Roudi` است.

بنابراین تمام Website Corpus در محدوده دانش مورد تأیید پروژه قرار دارد، اما سطح Attribution آن یکسان نیست.

### درجه ۱ — نویسندگی مستقیم

رکوردهای `primary_author` منابع درجه ۱ محسوب می‌شوند.

این منابع برای پاسخ‌هایی که مستقیماً به گفته، نوشته یا دیدگاه شخصی دکتر کمیل رودی نسبت داده می‌شوند، بالاترین اولویت را خواهند داشت.

### درجه ۲ — محتوای تحت نظارت

رکوردهای `supervised` منابع درجه ۲ محسوب می‌شوند.

این مقالات بخشی از Knowledge Corpus دکتر کمیل رودی هستند، زیرا تحت سردبیری، نظارت و تأیید ایشان قرار داشته‌اند؛ اما نباید به‌عنوان نوشته مستقیم ایشان معرفی شوند.

### نتیجه معماری

بنابراین سیستم آینده باید میان دو مفهوم زیر تفاوت قائل شود:

`knowledge_owner`

و

`direct_author`

در این Dataset، `knowledge_owner` برای تمام 529 مقاله دکتر کمیل رودی است، اما `direct_author` فقط برای 76 مقاله تأیید شده است.

این تفکیک باید در مراحل بعدی Retrieval، Citation و Generation حفظ شود.

</div>

<div style="direction: rtl; white-space: normal; line-height: 1;">

### اعتبارسنجی Contract مربوط به Attribution

اکنون که ساختار واقعی Attribution مشخص شده است، باید بررسی کنیم آیا تمام 529 رکورد دقیقاً از این قرارداد پیروی می‌کنند یا خیر.

دو وضعیت معتبر برای Website Corpus تعریف می‌شود:

**حالت اول — Primary Author**

- `content_origin = primary_author`
- `author = Dr. Komeil Roudi`
- `knowledge_owner = Dr. Komeil Roudi`

**حالت دوم — Supervised**

- `content_origin = supervised`
- نویسنده مستقیم ثبت نشده است.
- `knowledge_owner = Dr. Komeil Roudi`

هر رکوردی که خارج از این دو حالت باشد باید به‌عنوان Attribution Anomaly ثبت شود و بدون بررسی انسانی پذیرفته نشود.

همچنین تعداد دو گروه با وضعیت واقعی Dataset مقایسه خواهد شد.

**معیار PASS:**

- 76 رکورد `primary_author` وجود داشته باشد.
- 453 رکورد `supervised` وجود داشته باشد.
- تمام رکوردهای `primary_author` دارای نویسنده مستقیم دکتر کمیل رودی باشند.
- تمام رکوردهای `supervised` فاقد انتساب مستقیم نویسندگی به ایشان باشند.
- `knowledge_owner` در هر 529 رکورد برابر دکتر کمیل رودی باشد.
- هیچ Attribution Anomaly وجود نداشته باشد.

</div>

In [4]:
# Code Cell 04

expected_knowledge_owner = "Dr. Komeil Roudi"
expected_direct_author = "Dr. Komeil Roudi"

primary_author_records = []
supervised_records = []
attribution_anomalies = []

for record in website_manifest:
    content_origin = record.get("content_origin")
    author = record.get("author")
    knowledge_owner = record.get("knowledge_owner")

    if content_origin == "primary_author":
        primary_author_records.append(record)

        valid = (
            author == expected_direct_author
            and knowledge_owner == expected_knowledge_owner
        )

        if not valid:
            attribution_anomalies.append(
                {
                    "article_id": record.get("article_id"),
                    "content_origin": content_origin,
                    "author": author,
                    "knowledge_owner": knowledge_owner,
                }
            )

    elif content_origin == "supervised":
        supervised_records.append(record)

        valid = (
            author is None
            and knowledge_owner == expected_knowledge_owner
        )

        if not valid:
            attribution_anomalies.append(
                {
                    "article_id": record.get("article_id"),
                    "content_origin": content_origin,
                    "author": author,
                    "knowledge_owner": knowledge_owner,
                }
            )

    else:
        attribution_anomalies.append(
            {
                "article_id": record.get("article_id"),
                "content_origin": content_origin,
                "author": author,
                "knowledge_owner": knowledge_owner,
            }
        )

print("Primary-author records:", len(primary_author_records))
print("Supervised records:", len(supervised_records))
print("Attribution anomalies:", len(attribution_anomalies))

assert len(primary_author_records) == 76
assert len(supervised_records) == 453
assert len(attribution_anomalies) == 0

assert (
    len(primary_author_records) + len(supervised_records)
    == len(website_manifest)
)

print("\nWEBSITE ATTRIBUTION CONTRACT GATE: PASS")

Primary-author records: 76
Supervised records: 453
Attribution anomalies: 0

WEBSITE ATTRIBUTION CONTRACT GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### نتیجه اعتبارسنجی Attribution

تمام 529 رکورد Website با Contract تعریف‌شده برای Attribution سازگار هستند.

نتیجه واقعی Dataset:

- 76 رکورد `primary_author`
- 453 رکورد `supervised`
- صفر Attribution Anomaly

بنابراین در وضعیت فعلی Manifest، هیچ رکوردی مشاهده نشد که انتساب نویسنده، منشأ محتوا و مالک دانش آن با مدل تعریف‌شده پروژه ناسازگار باشد.

از این مرحله به بعد این تفکیک به‌عنوان بخشی از Data Contract Website حفظ خواهد شد.

</div>

<div style="direction: rtl; white-space: normal; line-height: 1;">

### بررسی هویت رکوردها، Stable ID و URL

هر مقاله در Knowledge Base باید یک هویت پایدار و قابل ردیابی داشته باشد.

اگر Stable ID تکراری باشد یا URLهای اصلی ناقص باشند، بعداً در Lineage، Citation، Deduplication و Retrieval با مشکل مواجه خواهیم شد.

در این مرحله چهار فیلد اصلی هویتی بررسی می‌شوند:

- `article_id`
- `source_id`
- `url`
- `canonical_url`

ابتدا وضعیت واقعی این فیلدها بررسی می‌شود و موارد زیر اندازه‌گیری خواهند شد:

- تعداد مقادیر خالی
- تعداد مقادیر Unique
- تعداد Duplicateها
- تعداد URLهای نامعتبر
- میزان تفاوت میان `url` و `canonical_url`

### نکته مهم

در این مرحله فرض نمی‌کنیم که `source_id` الزاماً باید برای هر مقاله Unique باشد؛ ابتدا رفتار واقعی آن را مشاهده می‌کنیم.

اما `article_id` به‌عنوان Stable ID هر مقاله باید در تمام Dataset یکتا و غیرخالی باشد.

همچنین `canonical_url` باید برای هر مقاله معتبر و قابل استفاده برای Lineage و Citation باشد.

**معیار PASS اصلی این مرحله:**

- تمام `article_id`ها غیرخالی باشند.
- Duplicate `article_id` برابر صفر باشد.
- تمام `url`ها و `canonical_url`ها غیرخالی باشند.
- URLها ساختار معتبر `http` یا `https` داشته باشند.

نتایج `source_id` و رابطه `url` با `canonical_url` نیز ثبت می‌شوند تا در صورت نیاز در مرحله بعد تحلیل شوند.

</div>

In [5]:
# Code Cell 05

from urllib.parse import urlparse

identity_fields = [
    "article_id",
    "source_id",
    "url",
    "canonical_url",
]

identity_summary = {}

for field_name in identity_fields:
    values = [record.get(field_name) for record in website_manifest]

    empty_count = sum(
        1
        for value in values
        if value is None
        or (isinstance(value, str) and not value.strip())
    )

    non_empty_values = [
        value.strip() if isinstance(value, str) else value
        for value in values
        if value is not None
        and not (isinstance(value, str) and not value.strip())
    ]

    unique_count = len(set(non_empty_values))
    duplicate_count = len(non_empty_values) - unique_count

    identity_summary[field_name] = {
        "empty_count": empty_count,
        "unique_count": unique_count,
        "duplicate_count": duplicate_count,
    }

    print(
        f"{field_name:<15} "
        f"empty={empty_count:<3} "
        f"unique={unique_count:<3} "
        f"duplicates={duplicate_count:<3}"
    )

invalid_urls = []
invalid_canonical_urls = []

for record in website_manifest:
    for field_name, error_bucket in [
        ("url", invalid_urls),
        ("canonical_url", invalid_canonical_urls),
    ]:
        value = record.get(field_name)

        if not isinstance(value, str):
            error_bucket.append(record.get("article_id"))
            continue

        parsed = urlparse(value.strip())

        if parsed.scheme not in {"http", "https"} or not parsed.netloc:
            error_bucket.append(record.get("article_id"))

url_canonical_mismatches = [
    record.get("article_id")
    for record in website_manifest
    if record.get("url") != record.get("canonical_url")
]

print("\nInvalid url values:", len(invalid_urls))
print("Invalid canonical_url values:", len(invalid_canonical_urls))
print(
    "url != canonical_url:",
    len(url_canonical_mismatches),
)

assert identity_summary["article_id"]["empty_count"] == 0
assert identity_summary["article_id"]["duplicate_count"] == 0

assert identity_summary["url"]["empty_count"] == 0
assert identity_summary["canonical_url"]["empty_count"] == 0

assert len(invalid_urls) == 0
assert len(invalid_canonical_urls) == 0

print("\nWEBSITE IDENTITY AND URL GATE: PASS")

article_id      empty=0   unique=529 duplicates=0  
source_id       empty=0   unique=529 duplicates=0  
url             empty=0   unique=529 duplicates=0  
canonical_url   empty=0   unique=529 duplicates=0  

Invalid url values: 0
Invalid canonical_url values: 0
url != canonical_url: 529

WEBSITE IDENTITY AND URL GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### تفسیر هویت و URL رکوردهای Website

تمام 529 رکورد دارای هویت یکتا هستند.

هیچ Duplicate در `article_id`، `source_id`، `url` یا `canonical_url` مشاهده نشد و تمام URLها از نظر ساختار اولیه معتبر هستند.

در عین حال، در تمام 529 رکورد مقدار `url` با `canonical_url` متفاوت است.

این تفاوت در این مرحله به‌عنوان خطا در نظر گرفته نمی‌شود، زیرا هر دو فیلد معتبر و یکتا هستند.

اما قبل از ادامه Acceptance باید مشخص شود که این اختلاف ناشی از چه نوع Transformation یا Normalization است.

بنابراین در مرحله بعد رابطه واقعی میان `url` و `canonical_url` بررسی خواهد شد و تا قبل از مشاهده شواهد، هیچ فرضی درباره علت اختلاف این دو فیلد پذیرفته نمی‌شود.

</div>

<div style="direction: rtl; white-space: normal; line-height: 1;">

### تحلیل رابطه میان `url` و `canonical_url`

در این مرحله تفاوت میان URL کشف‌شده و Canonical URL به‌صورت واقعی بررسی می‌شود.

هدف این است که مشخص شود آیا تفاوت‌ها فقط مربوط به مواردی مانند:

- تغییر Scheme
- تغییر Host
- حذف یا اضافه شدن `www`
- تغییر Path
- Query Parameter
- Fragment
- Slash انتهایی

هستند یا الگوی دیگری وجود دارد.

همچنین چند نمونه واقعی از زوج‌های `url` و `canonical_url` نمایش داده می‌شود تا بررسی انسانی نیز ممکن باشد.

این مرحله فقط تحلیل می‌کند و هیچ URLی را تغییر نمی‌دهد.

**معیار PASS:**  
تحلیل برای تمام 529 رکورد انجام شود و نوع اختلاف‌ها قابل اندازه‌گیری باشد.

</div>

In [6]:
# Code Cell 06

url_difference_summary = Counter()
url_pair_samples = []

for record in website_manifest:
    original_url = record["url"]
    canonical_url = record["canonical_url"]

    original = urlparse(original_url)
    canonical = urlparse(canonical_url)

    differences = []

    if original.scheme != canonical.scheme:
        differences.append("scheme")

    if original.netloc != canonical.netloc:
        differences.append("host")

    if original.path != canonical.path:
        differences.append("path")

    if original.query != canonical.query:
        differences.append("query")

    if original.fragment != canonical.fragment:
        differences.append("fragment")

    if not differences:
        differences.append("other_string_difference")

    difference_key = "+".join(differences)
    url_difference_summary[difference_key] += 1

    if len(url_pair_samples) < 10:
        url_pair_samples.append(
            {
                "article_id": record["article_id"],
                "url": original_url,
                "canonical_url": canonical_url,
                "difference": difference_key,
            }
        )

print("URL difference patterns:")

for pattern, count in url_difference_summary.most_common():
    print(f"  {pattern}: {count}")

print("\nSample URL pairs:")

for sample in url_pair_samples:
    print("\nArticle ID:", sample["article_id"])
    print("URL:", sample["url"])
    print("Canonical:", sample["canonical_url"])
    print("Difference:", sample["difference"])

assert sum(url_difference_summary.values()) == len(website_manifest)

print("\nURL RELATIONSHIP ANALYSIS GATE: PASS")

URL difference patterns:
  path: 529

Sample URL pairs:

Article ID: 008E61CC
URL: https://www.fintelligence.ir/Blog/Detail/008E61CC/%D8%B9%D9%84%D9%81-%D8%AF%D8%A7%D8%AF%D9%85-%D8%B4%DB%8C%D8%B1-%DA%AF%D8%B1%D9%81%D8%AA%D9%85
Canonical: https://www.fintelligence.ir/Blog/Detail/008E61CC
Difference: path

Article ID: 00D061A8
URL: https://www.fintelligence.ir/Blog/Detail/00D061A8/%D9%88%D9%82%D8%A7%DB%8C%D8%B9-%D9%86%DA%AF%D8%A7%D8%B1%DB%8C-%D9%87%D8%A7%DB%8C-%D8%B3%D9%86%D8%AC%D8%B4-%D8%B3%D9%88%D8%A7%D8%AF-%D9%85%D8%A7%D9%84%DB%8C-10
Canonical: https://www.fintelligence.ir/Blog/Detail/00D061A8
Difference: path

Article ID: 00D9A87F
URL: https://www.fintelligence.ir/Blog/Detail/00D9A87F/%DA%86%D8%B7%D9%88%D8%B1-%D8%B4%D8%BA%D9%84%D9%85%D8%A7%D9%86-%D8%B1%D8%A7-%D8%AA%D9%88%D8%B3%D8%B9%D9%87-%D8%AF%D9%87%DB%8C%D9%85
Canonical: https://www.fintelligence.ir/Blog/Detail/00D9A87F
Difference: path

Article ID: 00EDA828
URL: https://www.fintelligence.ir/Blog/Detail/00EDA828/%D9%88%D9%82%D8%A7

<div style="direction: rtl; white-space: normal; line-height: 1;">

### تفسیر رابطه URL و Canonical URL

تحلیل هر 529 رکورد نشان داد تفاوت میان `url` و `canonical_url` فقط در بخش `path` قرار دارد.

در URL اصلی، بعد از شناسه مقاله یک Slug متنی مرتبط با عنوان مقاله وجود دارد.

برای مثال:

`https://www.fintelligence.ir/Blog/Detail/008E61CC/<article-slug>`

در مقابل، Canonical URL فقط تا شناسه پایدار مقاله ادامه دارد:

`https://www.fintelligence.ir/Blog/Detail/008E61CC`

این الگو در هر 529 رکورد مشاهده شد.

بنابراین اختلاف میان این دو فیلد در وضعیت فعلی Dataset به‌عنوان Anomaly در نظر گرفته نمی‌شود.

`url` آدرس کامل صفحه را حفظ می‌کند، در حالی که `canonical_url` می‌تواند به‌عنوان آدرس پایدارتر مبتنی بر شناسه مقاله برای Lineage و Citation مورد استفاده قرار گیرد.

هر دو مقدار باید در Dataset حفظ شوند و یکی جایگزین دیگری نشود.

</div>

<div style="direction: rtl; white-space: normal; line-height: 1;">

### بررسی ارتباط Stable ID با Source ID و Canonical URL

در مرحله قبل مشخص شد Canonical URL هر مقاله بر اساس یک شناسه کوتاه در مسیر صفحه ساخته شده است.

اکنون باید بررسی کنیم آیا این شناسه واقعاً با `article_id` ذخیره‌شده در Manifest یکسان است یا خیر.

همچنین رابطه `source_id` با `article_id` بررسی می‌شود.

این موضوع برای Lineage بسیار مهم است؛ زیرا اگر شناسه ذخیره‌شده با شناسه واقعی داخل URL هماهنگ نباشد، اتصال Document به Source اصلی قابل اعتماد نخواهد بود.

در این مرحله موارد زیر اندازه‌گیری می‌شوند:

- تعداد رکوردهایی که `article_id` در انتهای Canonical URL قرار دارد.
- تعداد ناسازگاری‌ها میان `article_id` و Canonical URL.
- تعداد رکوردهایی که `source_id` با `article_id` برابر است.
- چند نمونه واقعی از رابطه `article_id` و `source_id`.

در این مرحله درباره ساختار `source_id` فرض قبلی نداریم و ابتدا رفتار واقعی Dataset را مشاهده می‌کنیم.

**معیار PASS اصلی:**

تمام 529 Canonical URL باید به `article_id` همان رکورد منتهی شوند.

</div>

In [7]:
# Code Cell 07

canonical_id_mismatches = []
source_id_equals_article_id = 0
source_id_samples = []

for record in website_manifest:
    article_id = record["article_id"]
    source_id = record["source_id"]
    canonical_url = record["canonical_url"]

    canonical_path = urlparse(canonical_url).path.rstrip("/")
    canonical_tail = canonical_path.split("/")[-1]

    if canonical_tail != article_id:
        canonical_id_mismatches.append(
            {
                "article_id": article_id,
                "canonical_tail": canonical_tail,
                "canonical_url": canonical_url,
            }
        )

    if source_id == article_id:
        source_id_equals_article_id += 1

    if len(source_id_samples) < 10:
        source_id_samples.append(
            {
                "article_id": article_id,
                "source_id": source_id,
            }
        )

print("Canonical URL / article_id mismatches:", len(canonical_id_mismatches))
print(
    "source_id == article_id:",
    source_id_equals_article_id,
    "/",
    len(website_manifest),
)

print("\nSample ID relationships:")

for sample in source_id_samples:
    print(
        f"  article_id={sample['article_id']} | "
        f"source_id={sample['source_id']}"
    )

assert len(canonical_id_mismatches) == 0

print("\nWEBSITE STABLE ID RELATIONSHIP GATE: PASS")

Canonical URL / article_id mismatches: 0
source_id == article_id: 0 / 529

Sample ID relationships:
  article_id=008E61CC | source_id=website:008E61CC
  article_id=00D061A8 | source_id=website:00D061A8
  article_id=00D9A87F | source_id=website:00D9A87F
  article_id=00EDA828 | source_id=website:00EDA828
  article_id=0152EE71 | source_id=website:0152EE71
  article_id=01AE96B5 | source_id=website:01AE96B5
  article_id=01D66D39 | source_id=website:01D66D39
  article_id=021FED66 | source_id=website:021FED66
  article_id=02376891 | source_id=website:02376891
  article_id=0248318E | source_id=website:0248318E

WEBSITE STABLE ID RELATIONSHIP GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### تفسیر ساختار Stable ID

بررسی تمام 529 رکورد نشان داد `article_id` با شناسه موجود در انتهای `canonical_url` کاملاً سازگار است و هیچ ناسازگاری مشاهده نشد.

همچنین مشخص شد `source_id` عمداً با `article_id` یکسان نیست.

ساختار واقعی آن به‌صورت Namespaced است:

`website:<article_id>`

برای مثال:

`website:008E61CC`

این ساختار باعث می‌شود شناسه یک رکورد علاوه بر هویت محلی مقاله، نوع منبع را نیز مشخص کند.

این موضوع برای Dataset چندمنبعی فاز دوم مفید است، زیرا از Collision میان شناسه‌های Website، Book و Transcript جلوگیری می‌کند.

در مرحله بعد این الگو برای تمام 529 رکورد به‌صورت رسمی اعتبارسنجی می‌شود.

</div>

<div style="direction: rtl; white-space: normal; line-height: 1;">

### اعتبارسنجی Contract مربوط به `source_id`

بر اساس ساختار واقعی Dataset، انتظار داریم `source_id` هر مقاله دقیقاً از ترکیب نوع منبع و `article_id` ساخته شده باشد.

الگوی مورد انتظار:

`website:<article_id>`

برای مثال:

`website:008E61CC`

این Validation بررسی می‌کند که این رابطه در تمام 529 رکورد بدون استثنا برقرار باشد.

**معیار PASS:**

- تمام `source_id`ها دقیقاً از الگوی `website:<article_id>` پیروی کنند.
- هیچ ناسازگاری وجود نداشته باشد.
- تمام `source_id`ها Unique باقی بمانند.

وجود حتی یک ناسازگاری به معنی مشکل در Stable Identity یا Lineage آن رکورد خواهد بود.

</div>

In [8]:
# Code Cell 08

source_id_mismatches = []

for record in website_manifest:
    article_id = record["article_id"]
    source_id = record["source_id"]

    expected_source_id = f"website:{article_id}"

    if source_id != expected_source_id:
        source_id_mismatches.append(
            {
                "article_id": article_id,
                "source_id": source_id,
                "expected_source_id": expected_source_id,
            }
        )

source_ids = [
    record["source_id"]
    for record in website_manifest
]

unique_source_ids = len(set(source_ids))

print("Total source IDs:", len(source_ids))
print("Unique source IDs:", unique_source_ids)
print("Source ID contract mismatches:", len(source_id_mismatches))

assert unique_source_ids == len(website_manifest)
assert len(source_id_mismatches) == 0

print("\nWEBSITE SOURCE ID CONTRACT GATE: PASS")

Total source IDs: 529
Unique source IDs: 529
Source ID contract mismatches: 0

WEBSITE SOURCE ID CONTRACT GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### نتیجه اعتبارسنجی Stable Source ID

تمام 529 رکورد دارای `source_id` یکتا و سازگار با Contract تعریف‌شده هستند.

ساختار تمام شناسه‌ها دقیقاً از الگوی زیر پیروی می‌کند:

`website:<article_id>`

هیچ Collision یا ناسازگاری در این بخش مشاهده نشد.

بنابراین هویت منطقی رکوردهای Website در سطح Manifest قابل قبول است.

مرحله بعد بررسی خواهد کرد که آیا این هویت منطقی به Artifactهای واقعی استخراج‌شده از Website نیز قابل اتصال است یا خیر.

</div>

<div style="direction: rtl; white-space: normal; line-height: 1;">

### کشف Artifactهای Persist‌شده مربوط به Website

Manifest فعلی هویت 529 مقاله را ثبت می‌کند، اما برای Lineage کامل کافی نیست که فقط بدانیم یک مقاله چه `article_id` یا URLی دارد.

باید مشخص شود خروجی واقعی فرآیند Ingestion در چه مسیرهایی ذخیره شده است و چه نوع فایل‌هایی برای هر مقاله وجود دارند.

در این مرحله هنوز هیچ ساختار از پیش تعریف‌شده‌ای مانند `raw`، `processed` یا `articles` را الزامی فرض نمی‌کنیم.

ابتدا ساختار واقعی موجود در `data/phase2` بررسی می‌شود.

برای جلوگیری از خروجی بسیار بزرگ، به‌جای چاپ تمام فایل‌ها موارد زیر نمایش داده می‌شوند:

- پوشه‌های سطح اول `data/phase2`
- تعداد کل فایل‌های موجود در هر پوشه
- پسوند فایل‌های موجود و تعداد هر نوع
- حداکثر 10 نمونه مسیر از هر بخش

هدف این مرحله فقط کشف ساختار واقعی Artifactها است و هیچ فایل یا پوشه‌ای ایجاد، حذف یا تغییر داده نمی‌شود.

**معیار PASS:**

عملیات Inventory بدون خطا انجام شود و ساختار واقعی Artifactهای موجود قابل مشاهده باشد.

وجود یا نبود Artifactهای مورد نیاز Lineage در مرحله بعد بر اساس همین نتیجه ارزیابی خواهد شد.

</div>

In [9]:
# Code Cell 09

from collections import Counter

phase2_data_root = project_root / "data" / "phase2"

assert phase2_data_root.is_dir(), (
    f"Phase 2 data directory not found: {phase2_data_root}"
)

top_level_entries = sorted(
    phase2_data_root.iterdir(),
    key=lambda path: path.name.lower(),
)

print("Phase 2 data root:")
print(phase2_data_root.relative_to(project_root))

print("\nTop-level entries:")

for entry in top_level_entries:
    entry_type = "DIR" if entry.is_dir() else "FILE"
    print(f"  [{entry_type}] {entry.name}")

print("\nArtifact inventory:")

for entry in top_level_entries:
    if not entry.is_dir():
        continue

    files = sorted(
        path
        for path in entry.rglob("*")
        if path.is_file()
    )

    extension_counts = Counter(
        path.suffix.lower() if path.suffix else "<no_extension>"
        for path in files
    )

    print(f"\n[{entry.name}]")
    print("Total files:", len(files))

    if extension_counts:
        print("Extensions:")
        for extension, count in extension_counts.most_common():
            print(f"  {extension}: {count}")

    if files:
        print("Sample paths:")
        for path in files[:10]:
            print(
                "  -",
                path.relative_to(project_root)
            )

print("\nWEBSITE ARTIFACT DISCOVERY GATE: PASS")

Phase 2 data root:
data/phase2

Top-level entries:
  [DIR] chunks
  [DIR] evaluation
  [DIR] interim
  [DIR] manifests
  [DIR] processed
  [DIR] raw

Artifact inventory:

[chunks]
Total files: 1
Extensions:
  .json: 1
Sample paths:
  - data/phase2/chunks/website_chunks_v2.json

[evaluation]
Total files: 0

[interim]
Total files: 1
Extensions:
  .json: 1
Sample paths:
  - data/phase2/interim/website_extracted_v2.json

[manifests]
Total files: 1
Extensions:
  .json: 1
Sample paths:
  - data/phase2/manifests/website_manifest_v1.json

[processed]
Total files: 1
Extensions:
  .json: 1
Sample paths:
  - data/phase2/processed/website_documents_v1.json

[raw]
Total files: 1060
Extensions:
  .html: 530
  .json: 530
Sample paths:
  - data/phase2/raw/website/008E61CC/article.html
  - data/phase2/raw/website/008E61CC/source.json
  - data/phase2/raw/website/00D061A8/article.html
  - data/phase2/raw/website/00D061A8/source.json
  - data/phase2/raw/website/00D9A87F/article.html
  - data/phase2/raw/we

<div style="direction: rtl; white-space: normal; line-height: 1;">

### تفسیر ساختار Artifactهای Website

بررسی `data/phase2` نشان داد Pipeline فعلی Website چند Artifact Persist‌شده تولید کرده است:

- Raw Snapshotها
- داده استخراج‌شده میانی
- Documentهای پردازش‌شده
- Chunkها
- Manifest

در مسیر `raw` تعداد 530 فایل HTML و 530 فایل JSON مشاهده شد، در حالی که Manifest شامل 529 رکورد است.

این اختلاف یک رکوردی هنوز به‌عنوان خطا یا Duplicate تفسیر نمی‌شود.

ممکن است Artifact اضافی مربوط به یک رکورد خارج از Manifest، اجرای آزمایشی، رکورد حذف‌شده یا وضعیت دیگری باشد.

در مرحله بعد شناسه‌های واقعی موجود در Raw با 529 `article_id` موجود در Manifest مقایسه می‌شوند تا Missing و Extra Artifactها به‌صورت دقیق مشخص شوند.

</div>

<div style="direction: rtl; white-space: normal; line-height: 1;">

### تطبیق Manifest با Raw Artifactهای Website

در این مرحله ارتباط میان 529 رکورد Manifest و Snapshotهای خام ذخیره‌شده روی Disk بررسی می‌شود.

برای هر `article_id` انتظار داریم یک پوشه متناظر در مسیر زیر وجود داشته باشد:

`data/phase2/raw/website/<article_id>/`

و در هر پوشه حداقل دو Artifact اصلی وجود داشته باشد:

- `article.html`
- `source.json`

موارد زیر اندازه‌گیری خواهند شد:

- تعداد شناسه‌های موجود در Manifest
- تعداد پوشه‌های Raw
- شناسه‌های Manifest که Raw ندارند
- شناسه‌های Raw که در Manifest وجود ندارند
- تعداد رکوردهایی که `article.html` ندارند
- تعداد رکوردهایی که `source.json` ندارند

### معیار اصلی Acceptance

برای تمام رکوردهای موجود در Manifest باید Raw Artifact کامل وجود داشته باشد.

بنابراین:

- Missing Raw برای رکوردهای Manifest باید صفر باشد.
- Missing `article.html` باید صفر باشد.
- Missing `source.json` باید صفر باشد.

وجود Raw اضافی الزاماً باعث Fail فوری نمی‌شود، اما باید شناسایی و علت آن در ادامه بررسی شود.

</div>

In [11]:
# Code Cell 10

raw_website_root = (
    project_root
    / "data"
    / "phase2"
    / "raw"
    / "website"
)

assert raw_website_root.is_dir(), (
    f"Raw website directory not found: {raw_website_root}"
)

manifest_article_ids = {
    record["article_id"]
    for record in website_manifest
}

raw_article_directories = sorted(
    path
    for path in raw_website_root.iterdir()
    if path.is_dir()
)

raw_article_ids = {
    path.name
    for path in raw_article_directories
}

missing_raw_ids = sorted(
    manifest_article_ids - raw_article_ids
)

extra_raw_ids = sorted(
    raw_article_ids - manifest_article_ids
)

missing_html_ids = []
missing_source_json_ids = []

for article_id in sorted(manifest_article_ids):
    article_directory = raw_website_root / article_id

    if not (article_directory / "article.html").is_file():
        missing_html_ids.append(article_id)

    if not (article_directory / "source.json").is_file():
        missing_source_json_ids.append(article_id)

print("Manifest article IDs:", len(manifest_article_ids))
print("Raw article directories:", len(raw_article_ids))

print("\nMissing Raw IDs:", len(missing_raw_ids))
for article_id in missing_raw_ids[:20]:
    print("  -", article_id)

print("\nExtra Raw IDs:", len(extra_raw_ids))
for article_id in extra_raw_ids[:20]:
    print("  -", article_id)

print("\nMissing article.html:", len(missing_html_ids))
print("Missing source.json:", len(missing_source_json_ids))

assert len(manifest_article_ids) == 529
assert len(missing_raw_ids) == 0
assert len(missing_html_ids) == 0
assert len(missing_source_json_ids) == 0

print("\nMANIFEST TO RAW COVERAGE GATE: PASS")

Manifest article IDs: 529
Raw article directories: 530

Missing Raw IDs: 0

Extra Raw IDs: 1
  - TEST_INVALID

Missing article.html: 0
Missing source.json: 0

MANIFEST TO RAW COVERAGE GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### تفسیر پوشش Raw Artifactها

تمام 529 رکورد موجود در Manifest دارای پوشه Raw متناظر هستند و برای همه آن‌ها هر دو فایل اصلی زیر وجود دارد:

- `article.html`
- `source.json`

بنابراین از نظر پوشش Raw، هیچ رکورد Manifest بدون Snapshot خام وجود ندارد.

در عین حال یک پوشه Raw اضافی با شناسه زیر شناسایی شد:

`TEST_INVALID`

این پوشه در Manifest حضور ندارد.

صرف نام این پوشه برای نتیجه‌گیری کافی نیست و هنوز مشخص نیست که یک Fixture آزمایشی، Artifact تست یا داده خارج از Dataset اصلی است.

در مرحله بعد محتویات همین Raw اضافی بررسی می‌شود تا نقش واقعی آن مشخص شود.

</div>

<div style="direction: rtl; white-space: normal; line-height: 1;">

### بررسی Raw Artifact اضافی خارج از Manifest

در این مرحله فقط Artifact اضافی `TEST_INVALID` بررسی می‌شود.

هدف این است که بدون تغییر فایل‌ها مشخص کنیم:

- چه فایل‌هایی داخل این پوشه وجود دارند.
- اندازه هر فایل چقدر است.
- `source.json` چه Metadataای دارد.
- آیا شواهدی وجود دارد که این Artifact عمداً برای تست یا Validation ساخته شده باشد.

این مرحله فقط Inspect انجام می‌دهد و هیچ فایل یا داده‌ای را حذف یا اصلاح نمی‌کند.

**معیار PASS:**

محتویات Artifact اضافی با موفقیت خوانده و نقش آن قابل بررسی باشد.

</div>

In [12]:
# Code Cell 11

extra_raw_inspection = {}

for extra_id in extra_raw_ids:
    extra_directory = raw_website_root / extra_id

    files = sorted(
        path
        for path in extra_directory.iterdir()
        if path.is_file()
    )

    file_info = {
        path.name: {
            "size_bytes": path.stat().st_size,
        }
        for path in files
    }

    source_json_path = extra_directory / "source.json"
    source_metadata = None

    if source_json_path.is_file():
        with source_json_path.open("r", encoding="utf-8") as file:
            source_metadata = json.load(file)

    extra_raw_inspection[extra_id] = {
        "files": file_info,
        "source_metadata": source_metadata,
    }

    print("Extra Raw ID:", extra_id)

    print("\nFiles:")
    for filename, info in file_info.items():
        print(
            f"  - {filename}: "
            f"{info['size_bytes']} bytes"
        )

    print("\nsource.json:")
    print(
        json.dumps(
            source_metadata,
            ensure_ascii=False,
            indent=2,
        )
    )

print("\nEXTRA RAW ARTIFACT INSPECTION GATE: PASS")

Extra Raw ID: TEST_INVALID

Files:
  - article.html: 47707 bytes
  - source.json: 501 bytes

source.json:
{
  "article_id": "TEST_INVALID",
  "url": "https://www.fintelligence.ir/Blog/Detail/INVALID_TEST_ID",
  "canonical_url": "https://www.fintelligence.ir/Blog/Detail/INVALID_TEST_ID",
  "http_status": 200,
  "content_type": "text/html; charset=utf-8",
  "crawled_at": "2026-08-06T03:21:39.588012+00:00",
  "raw_content_hash": "acfb1bcd818b802e991c9bff9b5e5b24fad038634c9cb3498ddc268e3cd1e332",
  "raw_size_bytes": 47707,
  "has_h1": false,
  "has_content_container": false,
  "is_valid_article": false
}

EXTRA RAW ARTIFACT INSPECTION GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### تفسیر Artifact اضافی `TEST_INVALID`

بررسی `TEST_INVALID` نشان داد این Artifact نباید به‌عنوان مقاله اضافی یا Duplicate در Website Corpus تفسیر شود.

Metadata ذخیره‌شده برای آن شامل شواهد زیر است:

- `is_valid_article = false`
- `has_h1 = false`
- `has_content_container = false`
- شناسه آن در Manifest اصلی حضور ندارد.
- URL آن شامل `INVALID_TEST_ID` است.

بنابراین این Artifact خارج از مجموعه 529 مقاله پذیرفته‌شده توسط Manifest قرار دارد.

در Acceptance فعلی:

- تعداد Raw Articleهای مربوط به Corpus اصلی همچنان 529 است.
- Missing Raw برابر صفر است.
- Artifact اضافی `TEST_INVALID` وارد Dataset پذیرفته‌شده نمی‌شود.
- این فایل در این Notebook حذف یا اصلاح نمی‌شود.

Metadata موجود به‌وضوح نشان می‌دهد این Snapshot به‌عنوان مقاله معتبر شناخته نشده است؛ با این حال چون فیلد مستقیمی برای اعلام نقش آن به‌عنوان Test Fixture وجود ندارد، این عنوان به‌عنوان یک استنباط فنی و نه Metadata صریح ثبت می‌شود.

</div>

<div style="direction: rtl; white-space: normal; line-height: 1;">

### کشف Schema واقعی Metadata خام Website

هر Snapshot خام Website علاوه بر فایل `article.html` دارای یک فایل `source.json` است.

این Metadata برای Lineage اهمیت زیادی دارد، زیرا می‌تواند اطلاعاتی مانند URL منبع، زمان Crawl، Hash محتوای خام، اندازه فایل و وضعیت اعتبار صفحه را نگهداری کند.

قبل از تعریف Validation نهایی، ابتدا Schema واقعی تمام 529 فایل `source.json` بررسی می‌شود.

در این مرحله موارد زیر استخراج خواهند شد:

- تمام فیلدهای موجود در `source.json`
- میزان Coverage هر فیلد در 529 مقاله
- نوع داده هر فیلد
- توزیع مقادیر مهمی مانند:
  - `http_status`
  - `is_valid_article`
  - `has_h1`
  - `has_content_container`

Artifact `TEST_INVALID` در این تحلیل وارد نمی‌شود، زیرا بخشی از Manifest اصلی 529 مقاله نیست.

**معیار PASS:**

- تمام 529 فایل `source.json` قابل خواندن باشند.
- تمام آن‌ها JSON Object معتبر باشند.
- Schema واقعی با موفقیت استخراج شود.

این مرحله هنوز صحت Hash یا محتوای HTML را تأیید نمی‌کند؛ آن Validation در مرحله بعد انجام خواهد شد.

</div>

In [13]:
# Code Cell 12

raw_source_records = []
raw_source_field_presence = Counter()
raw_source_field_types = defaultdict(Counter)

for article_id in sorted(manifest_article_ids):
    source_json_path = (
        raw_website_root
        / article_id
        / "source.json"
    )

    with source_json_path.open("r", encoding="utf-8") as file:
        source_metadata = json.load(file)

    assert isinstance(source_metadata, dict), (
        f"source.json is not a JSON object: {article_id}"
    )

    raw_source_records.append(source_metadata)

    for field_name, value in source_metadata.items():
        raw_source_field_presence[field_name] += 1
        raw_source_field_types[field_name][type(value).__name__] += 1

print("Raw source.json records:", len(raw_source_records))
print("Unique metadata fields:", len(raw_source_field_presence))

print("\nField coverage:")

for field_name in sorted(raw_source_field_presence):
    count = raw_source_field_presence[field_name]
    percentage = count / len(raw_source_records) * 100

    type_summary = ", ".join(
        f"{type_name}:{type_count}"
        for type_name, type_count
        in raw_source_field_types[field_name].items()
    )

    print(
        f"{field_name:<25} "
        f"{count:>3}/{len(raw_source_records)} "
        f"({percentage:6.2f}%) "
        f"types=[{type_summary}]"
    )

for field_name in [
    "http_status",
    "is_valid_article",
    "has_h1",
    "has_content_container",
]:
    distribution = Counter(
        record.get(field_name)
        for record in raw_source_records
    )

    print(f"\n{field_name} distribution:")
    for value, count in distribution.most_common():
        print(f"  {value!r}: {count}")

assert len(raw_source_records) == len(website_manifest)

print("\nRAW SOURCE METADATA SCHEMA GATE: PASS")

Raw source.json records: 529
Unique metadata fields: 11

Field coverage:
article_id                529/529 (100.00%) types=[str:529]
canonical_url             529/529 (100.00%) types=[str:529]
content_type              529/529 (100.00%) types=[str:529]
crawled_at                529/529 (100.00%) types=[str:529]
has_content_container     529/529 (100.00%) types=[bool:529]
has_h1                    529/529 (100.00%) types=[bool:529]
http_status               529/529 (100.00%) types=[int:529]
is_valid_article          529/529 (100.00%) types=[bool:529]
raw_content_hash          529/529 (100.00%) types=[str:529]
raw_size_bytes            529/529 (100.00%) types=[int:529]
url                       529/529 (100.00%) types=[str:529]

http_status distribution:
  200: 529

is_valid_article distribution:
  True: 529

has_h1 distribution:
  True: 529

has_content_container distribution:
  True: 529

RAW SOURCE METADATA SCHEMA GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### اعتبارسنجی Integrity فایل‌های خام Website

وجود فایل Raw به‌تنهایی کافی نیست.

باید مطمئن شویم فایل `article.html` که اکنون روی Disk وجود دارد دقیقاً همان محتوایی است که در زمان Crawl ثبت شده و بعداً تغییر، خراب یا جایگزین نشده است.

برای این کار، برای هر یک از 529 مقاله:

- فایل واقعی `article.html` دوباره خوانده می‌شود.
- SHA-256 آن از روی Bytes واقعی محاسبه می‌شود.
- Hash محاسبه‌شده با `raw_content_hash` موجود در `source.json` مقایسه می‌شود.
- اندازه واقعی فایل با `raw_size_bytes` ثبت‌شده مقایسه می‌شود.
- `article_id` موجود در `source.json` با پوشه و Manifest تطبیق داده می‌شود.
- `url` و `canonical_url` موجود در Raw Metadata با Manifest همان مقاله مقایسه می‌شوند.

این Validation یکی از پایه‌های اصلی Lineage است.

اگر Hash فایل خام با Metadata زمان Crawl هماهنگ نباشد، دیگر نمی‌توانیم با اطمینان بگوییم Documentهای مراحل بعدی از همان Source ثبت‌شده ساخته شده‌اند.

### معیار PASS

برای هر 529 مقاله:

- SHA-256 واقعی با `raw_content_hash` برابر باشد.
- اندازه واقعی فایل با `raw_size_bytes` برابر باشد.
- `article_id` سازگار باشد.
- `url` سازگار باشد.
- `canonical_url` سازگار باشد.

تعداد تمام ناسازگاری‌ها باید صفر باشد.

</div>

In [14]:
# Code Cell 13

manifest_by_article_id = {
    record["article_id"]: record
    for record in website_manifest
}

hash_mismatches = []
size_mismatches = []
article_id_mismatches = []
raw_url_mismatches = []
raw_canonical_url_mismatches = []

for article_id in sorted(manifest_article_ids):
    article_directory = raw_website_root / article_id
    html_path = article_directory / "article.html"
    source_json_path = article_directory / "source.json"

    html_bytes = html_path.read_bytes()
    actual_hash = hashlib.sha256(html_bytes).hexdigest()
    actual_size = len(html_bytes)

    with source_json_path.open("r", encoding="utf-8") as file:
        source_metadata = json.load(file)

    manifest_record = manifest_by_article_id[article_id]

    if actual_hash != source_metadata.get("raw_content_hash"):
        hash_mismatches.append(article_id)

    if actual_size != source_metadata.get("raw_size_bytes"):
        size_mismatches.append(article_id)

    if source_metadata.get("article_id") != article_id:
        article_id_mismatches.append(article_id)

    if source_metadata.get("url") != manifest_record.get("url"):
        raw_url_mismatches.append(article_id)

    if (
        source_metadata.get("canonical_url")
        != manifest_record.get("canonical_url")
    ):
        raw_canonical_url_mismatches.append(article_id)

print("Raw HTML files checked:", len(manifest_article_ids))
print("Hash mismatches:", len(hash_mismatches))
print("Size mismatches:", len(size_mismatches))
print("Article ID mismatches:", len(article_id_mismatches))
print("URL mismatches:", len(raw_url_mismatches))
print("Canonical URL mismatches:", len(raw_canonical_url_mismatches))

assert len(hash_mismatches) == 0
assert len(size_mismatches) == 0
assert len(article_id_mismatches) == 0
assert len(raw_url_mismatches) == 0
assert len(raw_canonical_url_mismatches) == 0

print("\nRAW INTEGRITY AND LINEAGE GATE: PASS")

Raw HTML files checked: 529
Hash mismatches: 0
Size mismatches: 0
Article ID mismatches: 0
URL mismatches: 0
Canonical URL mismatches: 0

RAW INTEGRITY AND LINEAGE GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### نتیجه اعتبارسنجی Raw Integrity و Lineage

تمام 529 فایل خام Website با Metadata زمان Crawl سازگار هستند.

نتایج Validation:

- Hash mismatch: صفر
- Size mismatch: صفر
- Article ID mismatch: صفر
- URL mismatch: صفر
- Canonical URL mismatch: صفر

بنابراین در این نقطه می‌توان با شواهد فنی تأیید کرد که فایل‌های خام موجود روی Disk همان Artifactهایی هستند که Metadata مربوط به آن‌ها هنگام Crawl ثبت شده است.

این نتیجه یکی از پایه‌های اصلی Lineage فاز دوم است.

زنجیره تأییدشده فعلی:

`Manifest → Raw Metadata → Raw HTML`

مرحله بعد بررسی می‌کند آیا خروجی استخراج‌شده میانی نیز برای همین مجموعه مقالات قابل ردیابی و سازگار است یا خیر.

</div>

<div style="direction: rtl; white-space: normal; line-height: 1;">

### کشف ساختار Interim Artifact مربوط به Website

پس از ذخیره Raw HTML، Pipeline Website محتوای صفحه را استخراج کرده و Artifact میانی زیر را تولید کرده است:

`data/phase2/interim/website_extracted_v2.json`

قبل از بررسی کیفیت یا Lineage این داده، ابتدا باید ساختار واقعی آن را بشناسیم.

در این مرحله بررسی می‌شود:

- فایل Interim وجود داشته باشد.
- JSON قابل خواندن باشد.
- نوع ساختار سطح اصلی مشخص شود.
- تعداد رکوردها اندازه‌گیری شود.
- Schema واقعی رکوردها استخراج شود.
- Coverage و نوع داده هر فیلد مشخص شود.

در این مرحله هیچ فرضی درباره تعداد رکوردها یا فیلدهای موجود نمی‌شود.

**معیار PASS:**

Artifact Interim قابل خواندن باشد و تمام رکوردهای آن ساختار قابل بررسی داشته باشند.

</div>

In [15]:
# Code Cell 14

interim_path = (
    project_root
    / "data"
    / "phase2"
    / "interim"
    / "website_extracted_v2.json"
)

assert interim_path.is_file(), (
    f"Interim artifact not found: {interim_path}"
)

with interim_path.open("r", encoding="utf-8") as file:
    interim_data = json.load(file)

print("Interim artifact:", interim_path.relative_to(project_root))
print("Top-level type:", type(interim_data).__name__)

if isinstance(interim_data, list):
    print("Record count:", len(interim_data))

    assert all(
        isinstance(record, dict)
        for record in interim_data
    ), "One or more interim records are not JSON objects."

    interim_field_presence = Counter()
    interim_field_types = defaultdict(Counter)

    for record in interim_data:
        for field_name, value in record.items():
            interim_field_presence[field_name] += 1
            interim_field_types[field_name][type(value).__name__] += 1

    print("\nField coverage:")

    for field_name in sorted(interim_field_presence):
        count = interim_field_presence[field_name]
        percentage = count / len(interim_data) * 100

        type_summary = ", ".join(
            f"{type_name}:{type_count}"
            for type_name, type_count
            in interim_field_types[field_name].items()
        )

        print(
            f"{field_name:<30} "
            f"{count:>3}/{len(interim_data)} "
            f"({percentage:6.2f}%) "
            f"types=[{type_summary}]"
        )

elif isinstance(interim_data, dict):
    print("Top-level keys:", sorted(interim_data.keys()))

else:
    raise TypeError(
        f"Unsupported interim JSON type: {type(interim_data).__name__}"
    )

print("\nINTERIM ARTIFACT DISCOVERY GATE: PASS")

Interim artifact: data/phase2/interim/website_extracted_v2.json
Top-level type: list
Record count: 529

Field coverage:
article_id                     529/529 (100.00%) types=[str:529]
author                         529/529 (100.00%) types=[str:529]
author_id                      529/529 (100.00%) types=[str:529]
body_text                      529/529 (100.00%) types=[str:529]
canonical_url                  529/529 (100.00%) types=[str:529]
content_origin                 529/529 (100.00%) types=[str:529]
knowledge_owner                529/529 (100.00%) types=[str:529]
raw_content_hash               529/529 (100.00%) types=[str:529]
source_id                      529/529 (100.00%) types=[str:529]
source_type                    529/529 (100.00%) types=[str:529]
study_time_minutes             529/529 (100.00%) types=[int:529]
tags                           529/529 (100.00%) types=[list:529]
title                          529/529 (100.00%) types=[str:529]
url                            529

<div style="direction: rtl; white-space: normal; line-height: 1;">

### تفسیر اولیه Interim Artifact

Artifact میانی `website_extracted_v2.json` شامل 529 رکورد است و تمام رکوردها دارای Schema یکسان با 15 فیلد هستند.

در این لایه علاوه بر شناسه‌ها و Metadata پایه، اطلاعات استخراج‌شده‌ای مانند عنوان، متن مقاله، Tagها، تعداد بازدید و زمان مطالعه نیز وجود دارد.

یک تفاوت مهم میان Manifest و Interim مشاهده شد:

در Manifest، فیلد `author` برای 453 مقاله `None` بود، اما در Interim فیلد `author` در هر 529 رکورد از نوع `str` است.

این تفاوت هنوز به‌عنوان خطا تفسیر نمی‌شود.

قبل از هر نتیجه‌گیری باید مقادیر واقعی `author` و `author_id` بررسی شوند تا مشخص شود Pipeline در مقالات `supervised` چه Metadataای ثبت کرده و آیا تفکیک نویسنده مستقیم و محتوای تحت نظارت همچنان حفظ شده است یا خیر.

</div>

<div style="direction: rtl; white-space: normal; line-height: 1;">

### بررسی Attribution در لایه Interim

Attribution در این پروژه یکی از بخش‌های حساس Data Contract است.

در Manifest مشخص شد:

- 76 مقاله `primary_author` هستند.
- 453 مقاله `supervised` هستند.
- تنها 76 مقاله دارای نویسنده مستقیم دکتر کمیل رودی هستند.

اکنون باید بررسی کنیم Pipeline استخراج در Interim این تفکیک را چگونه نگهداری کرده است.

در این مرحله توزیع مقادیر زیر بررسی می‌شود:

- `author`
- `author_id`
- `content_origin`
- `knowledge_owner`

همچنین ترکیب `content_origin` و `author` اندازه‌گیری می‌شود تا مشخص شود برای مقالات `supervised` چه مقداری در فیلد `author` قرار گرفته است.

هدف این مرحله کشف رفتار واقعی داده است و هنوز هیچ رکوردی اصلاح نمی‌شود.

**معیار PASS:**

- تمام 529 رکورد قابل طبقه‌بندی باشند.
- توزیع `content_origin` همچنان با 76 مقاله `primary_author` و 453 مقاله `supervised` سازگار باشد.
- `knowledge_owner` در تمام رکوردها دکتر کمیل رودی باقی مانده باشد.

نحوه مقداردهی `author` و `author_id` بر اساس خروجی واقعی تحلیل خواهد شد.

</div>

In [16]:
# Code Cell 15

interim_metadata_fields = [
    "author",
    "author_id",
    "content_origin",
    "knowledge_owner",
]

for field_name in interim_metadata_fields:
    distribution = Counter(
        record.get(field_name)
        for record in interim_data
    )

    print(f"\n{field_name}:")
    for value, count in distribution.most_common():
        print(f"  {value!r}: {count}")

author_by_origin = Counter(
    (
        record.get("content_origin"),
        record.get("author"),
    )
    for record in interim_data
)

print("\nAuthor by content_origin:")

for (content_origin, author), count in author_by_origin.most_common():
    print(
        f"  content_origin={content_origin!r} | "
        f"author={author!r} | "
        f"count={count}"
    )

interim_origin_distribution = Counter(
    record.get("content_origin")
    for record in interim_data
)

interim_knowledge_owner_distribution = Counter(
    record.get("knowledge_owner")
    for record in interim_data
)

assert interim_origin_distribution["primary_author"] == 76
assert interim_origin_distribution["supervised"] == 453

assert interim_knowledge_owner_distribution == Counter(
    {"Dr. Komeil Roudi": 529}
)

print("\nINTERIM ATTRIBUTION DISCOVERY GATE: PASS")


author:
  'دکتر کمیل رودی': 76
  'الهام میرمحمدی': 72
  'هادی نجف': 69
  'سجاد یزدانی': 51
  'عاطفه جلادتی': 35
  'سودابه فرخنده': 34
  'حمیده معماریان': 34
  'محمدرضا سیمیاری': 30
  'مریم محمدخواه': 28
  'دکتر نیره\u200cالسادات علوی': 19
  'مائده محبوب': 19
  'صاحب\u200cنظران': 16
  'شیما فدوی': 15
  'دکتر کوثر اخوان': 13
  'پژوهشگران\u200c آکادمی هوش مالی': 10
  'دکتر ماجد ناجی': 3
  'مهسا صادقی': 3
  'فاطمه قاسمی\u200cپور': 1
  'هستی غریبی': 1

author_id:
  '4': 76
  '2015': 72
  '10': 69
  '2018': 51
  '5': 35
  '6': 34
  '2011': 34
  '7': 30
  '11': 28
  '1011': 19
  '2016': 19
  '2013': 16
  '2014': 15
  '12': 13
  '2022': 10
  '2021': 3
  '2019': 3
  '2017': 1
  '2024': 1

content_origin:
  'supervised': 453
  'primary_author': 76

knowledge_owner:
  'Dr. Komeil Roudi': 529

Author by content_origin:
  content_origin='primary_author' | author='دکتر کمیل رودی' | count=76
  content_origin='supervised' | author='الهام میرمحمدی' | count=72
  content_origin='supervised' | author='ها

<div style="direction: rtl; white-space: normal; line-height: 1;">

### تفسیر Attribution در Interim

بررسی Interim نشان داد فیلد `author` در این لایه نسبت به Manifest اطلاعات دقیق‌تری دارد.

در Manifest، برای 453 مقاله `supervised` مقدار `author` برابر `None` بود؛ اما در Interim نویسنده مستقیم این مقالات از محتوای Website استخراج شده و به‌صورت صریح ثبت شده است.

نتیجه واقعی:

- 76 مقاله `primary_author` با نویسندگی مستقیم دکتر کمیل رودی
- 453 مقاله `supervised` با نویسندگان مستقیم دیگر
- تمام 529 مقاله دارای `knowledge_owner = Dr. Komeil Roudi`

بنابراین Interim تفکیک مهم میان سه مفهوم را حفظ کرده است:

- `author`: نویسنده مستقیم مقاله
- `content_origin`: نوع ارتباط مقاله با Corpus
- `knowledge_owner`: مالک یا مرجع دانش این مجموعه

این ساختار برای استفاده آینده در Retrieval و Generation مناسب‌تر از انتساب همه مقالات به یک نویسنده واحد است.

### قاعده کاربردی

اگر یک مقاله `supervised` باشد، سیستم می‌تواند آن را به‌عنوان بخشی از Knowledge Corpus دکتر کمیل رودی استفاده کند، اما نباید متن آن را به‌عنوان نوشته مستقیم ایشان معرفی کند.

برای پاسخ‌هایی که مشخصاً درباره «گفته یا نوشته مستقیم دکتر رودی» هستند، منابع `primary_author` باید مبنای اصلی باشند.

</div>

<div style="direction: rtl; white-space: normal; line-height: 1;">

### اعتبارسنجی Lineage میان Manifest و Interim

اکنون باید بررسی کنیم Artifact میانی دقیقاً متعلق به همان 529 مقاله ثبت‌شده در Manifest است.

برای این کار رکوردهای دو لایه بر اساس `article_id` به یکدیگر متصل می‌شوند و فیلدهای هویتی و Lineage مقایسه خواهند شد.

موارد زیر بررسی می‌شوند:

- مجموعه `article_id`ها در Manifest و Interim یکسان باشد.
- `source_id` در هر دو لایه برابر باشد.
- `url` برابر باشد.
- `canonical_url` برابر باشد.
- `content_origin` برابر باشد.
- `knowledge_owner` برابر باشد.
- `raw_content_hash` موجود در Interim با Hash ثبت‌شده در Raw Metadata همان مقاله برابر باشد.

### نکته مربوط به `author`

فیلد `author` مستقیماً میان Manifest و Interim مقایسه نمی‌شود، زیرا Manifest برای مقالات `supervised` مقدار `None` دارد ولی Interim نویسنده واقعی استخراج‌شده از صفحه را ثبت کرده است.

این تفاوت یک غنی‌سازی Metadata محسوب می‌شود، نه Lineage Failure.

**معیار PASS:**

تمام فیلدهای هویتی و Lineage در هر 529 رکورد بدون ناسازگاری باشند.

</div>

In [18]:
# Code Cell 16

interim_by_article_id = {
    record["article_id"]: record
    for record in interim_data
}

manifest_ids = set(manifest_by_article_id)
interim_ids = set(interim_by_article_id)

missing_in_interim = sorted(manifest_ids - interim_ids)
extra_in_interim = sorted(interim_ids - manifest_ids)

source_id_mismatches = []
url_mismatches = []
canonical_url_mismatches = []
content_origin_mismatches = []
knowledge_owner_mismatches = []
raw_hash_mismatches = []

raw_source_by_article_id = {
    record["article_id"]: record
    for record in raw_source_records
}

for article_id in sorted(manifest_ids & interim_ids):
    manifest_record = manifest_by_article_id[article_id]
    interim_record = interim_by_article_id[article_id]
    raw_source_record = raw_source_by_article_id[article_id]

    if manifest_record["source_id"] != interim_record["source_id"]:
        source_id_mismatches.append(article_id)

    if manifest_record["url"] != interim_record["url"]:
        url_mismatches.append(article_id)

    if manifest_record["canonical_url"] != interim_record["canonical_url"]:
        canonical_url_mismatches.append(article_id)

    if manifest_record["content_origin"] != interim_record["content_origin"]:
        content_origin_mismatches.append(article_id)

    if manifest_record["knowledge_owner"] != interim_record["knowledge_owner"]:
        knowledge_owner_mismatches.append(article_id)

    if interim_record["raw_content_hash"] != raw_source_record["raw_content_hash"]:
        raw_hash_mismatches.append(article_id)

print("Missing in Interim:", len(missing_in_interim))
print("Extra in Interim:", len(extra_in_interim))
print("Source ID mismatches:", len(source_id_mismatches))
print("URL mismatches:", len(url_mismatches))
print("Canonical URL mismatches:", len(canonical_url_mismatches))
print("Content origin mismatches:", len(content_origin_mismatches))
print("Knowledge owner mismatches:", len(knowledge_owner_mismatches))
print("Raw hash mismatches:", len(raw_hash_mismatches))

assert len(missing_in_interim) == 0
assert len(extra_in_interim) == 0
assert len(source_id_mismatches) == 0
assert len(url_mismatches) == 0
assert len(canonical_url_mismatches) == 0
assert len(content_origin_mismatches) == 0
assert len(knowledge_owner_mismatches) == 0
assert len(raw_hash_mismatches) == 0

print("\nMANIFEST TO INTERIM LINEAGE GATE: PASS")

Missing in Interim: 0
Extra in Interim: 0
Source ID mismatches: 0
URL mismatches: 0
Canonical URL mismatches: 0
Content origin mismatches: 0
Knowledge owner mismatches: 0
Raw hash mismatches: 0

MANIFEST TO INTERIM LINEAGE GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### نتیجه Lineage و بررسی اولیه کیفیت متن استخراج‌شده

اعتبارسنجی ارتباط میان Manifest و Interim با موفقیت انجام شد.

برای هر 529 مقاله:

- رکورد گم‌شده در Interim وجود ندارد.
- رکورد اضافه در Interim وجود ندارد.
- `source_id` بدون ناسازگاری منتقل شده است.
- `url` و `canonical_url` بدون ناسازگاری منتقل شده‌اند.
- `content_origin` حفظ شده است.
- `knowledge_owner` حفظ شده است.
- `raw_content_hash` با Metadata فایل خام همان مقاله سازگار است.

بنابراین زنجیره Lineage تا این مرحله قابل اثبات است:

`Manifest → Raw Metadata → Raw HTML → Interim`

اکنون باید از بررسی صرفاً ساختاری عبور کنیم و کیفیت واقعی متن استخراج‌شده را بررسی کنیم.

وجود 529 رکورد به‌تنهایی کافی نیست؛ ممکن است یک رکورد وجود داشته باشد اما عنوان یا متن آن خالی، بسیار کوتاه، تکراری یا حاصل Extraction ناقص باشد.

در این مرحله وضعیت واقعی دو فیلد اصلی محتوایی بررسی می‌شود:

- `title`
- `body_text`

موارد زیر اندازه‌گیری خواهند شد:

- تعداد عنوان‌های خالی
- تعداد متن‌های خالی
- حداقل، میانه و حداکثر طول عنوان
- حداقل، میانه و حداکثر طول متن
- تعداد متن‌های بسیار کوتاه
- تعداد عنوان‌های تکراری
- تعداد متن‌های کاملاً تکراری

Thresholdهای طول متن در این مرحله برای شناخت Dataset استفاده می‌شوند و هنوز به‌تنهایی باعث Reject شدن مقاله نمی‌شوند.

هدف این است که ابتدا Distribution واقعی محتوا را ببینیم و سپس بر اساس شواهد Dataset، Quality Gate مناسب تعریف کنیم.

**معیار PASS این مرحله:**

- هر 529 رکورد دارای `title` متنی و غیرخالی باشند.
- هر 529 رکورد دارای `body_text` متنی و غیرخالی باشند.
- تحلیل کیفیت بدون خطا برای کل Corpus انجام شود.

Duplicate یا متن بسیار کوتاه در این مرحله گزارش می‌شود و در صورت وجود، در مرحله بعد به‌صورت موردی بررسی خواهد شد.

</div>

In [19]:
# Code Cell 17

title_values = [
    record.get("title")
    for record in interim_data
]

body_values = [
    record.get("body_text")
    for record in interim_data
]

empty_title_ids = [
    record["article_id"]
    for record in interim_data
    if not isinstance(record.get("title"), str)
    or not record["title"].strip()
]

empty_body_ids = [
    record["article_id"]
    for record in interim_data
    if not isinstance(record.get("body_text"), str)
    or not record["body_text"].strip()
]

title_lengths = sorted(
    len(value.strip())
    for value in title_values
    if isinstance(value, str) and value.strip()
)

body_lengths = sorted(
    len(value.strip())
    for value in body_values
    if isinstance(value, str) and value.strip()
)

def median_from_sorted(values):
    count = len(values)
    middle = count // 2

    if count % 2:
        return values[middle]

    return (values[middle - 1] + values[middle]) / 2

short_body_thresholds = [100, 300, 500]

short_body_counts = {
    threshold: sum(
        1
        for length in body_lengths
        if length < threshold
    )
    for threshold in short_body_thresholds
}

normalized_titles = [
    record["title"].strip()
    for record in interim_data
    if isinstance(record.get("title"), str)
    and record["title"].strip()
]

normalized_bodies = [
    record["body_text"].strip()
    for record in interim_data
    if isinstance(record.get("body_text"), str)
    and record["body_text"].strip()
]

duplicate_title_count = (
    len(normalized_titles)
    - len(set(normalized_titles))
)

duplicate_body_count = (
    len(normalized_bodies)
    - len(set(normalized_bodies))
)

print("Total Interim records:", len(interim_data))
print("Empty titles:", len(empty_title_ids))
print("Empty body texts:", len(empty_body_ids))

print("\nTitle length statistics:")
print("  Minimum:", min(title_lengths))
print("  Median:", median_from_sorted(title_lengths))
print("  Maximum:", max(title_lengths))

print("\nBody length statistics:")
print("  Minimum:", min(body_lengths))
print("  Median:", median_from_sorted(body_lengths))
print("  Maximum:", max(body_lengths))

print("\nShort body counts:")
for threshold, count in short_body_counts.items():
    print(f"  < {threshold} characters: {count}")

print("\nExact duplicate counts:")
print("  Duplicate titles:", duplicate_title_count)
print("  Duplicate body texts:", duplicate_body_count)

assert len(empty_title_ids) == 0
assert len(empty_body_ids) == 0
assert len(title_lengths) == len(interim_data)
assert len(body_lengths) == len(interim_data)

print("\nINTERIM TEXT QUALITY DISCOVERY GATE: PASS")

Total Interim records: 529
Empty titles: 0
Empty body texts: 0

Title length statistics:
  Minimum: 4
  Median: 29
  Maximum: 81

Body length statistics:
  Minimum: 224
  Median: 1511
  Maximum: 9723

Short body counts:
  < 100 characters: 0
  < 300 characters: 3
  < 500 characters: 13

Exact duplicate counts:
  Duplicate titles: 26
  Duplicate body texts: 1

INTERIM TEXT QUALITY DISCOVERY GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### تفسیر کیفیت اولیه متن و بررسی موارد مشکوک

بررسی اولیه کیفیت متن نشان داد هر 529 رکورد Interim دارای عنوان و متن غیرخالی هستند.

نتایج اصلی:

- عنوان خالی: صفر
- متن خالی: صفر
- کوتاه‌ترین متن: 224 کاراکتر
- میانه طول متن: 1511 کاراکتر
- بلندترین متن: 9723 کاراکتر
- متن کمتر از 100 کاراکتر: صفر
- متن کمتر از 300 کاراکتر: 3 مورد
- متن کمتر از 500 کاراکتر: 13 مورد
- Duplicate دقیق عنوان: 26 مورد
- Duplicate دقیق متن: 1 مورد

وجود عنوان تکراری لزوماً به معنی Duplicate Document نیست.

ممکن است چند مقاله از یک مجموعه آموزشی، سری محتوایی یا موضوع مشترک عنوان یکسان داشته باشند اما متن و شناسه متفاوتی داشته باشند.

همچنین کوتاه بودن یک مقاله به‌تنهایی برای Reject کردن آن کافی نیست.

اما وجود یک Duplicate دقیق در `body_text` نیاز به بررسی دارد، زیرا ممکن است دو شناسه متفاوت عملاً محتوای یکسانی داشته باشند.

بنابراین قبل از تعریف Quality Gate نهایی، موارد مشکوک به‌صورت انسانی و قابل مشاهده بررسی می‌شوند.

در این مرحله موارد زیر نمایش داده خواهند شد:

- 15 مقاله با کوتاه‌ترین `body_text`
- گروه‌های دارای عنوان دقیقاً تکراری
- شناسه، URL و طول متن رکوردهای دارای عنوان تکراری
- گروه دارای `body_text` کاملاً یکسان

در این مرحله هیچ رکوردی حذف یا Reject نمی‌شود.

هدف تشخیص تفاوت میان Duplicate واقعی و شباهت طبیعی محتوایی است.

**معیار PASS:**

تمام موارد مشکوک قابل شناسایی و قابل ردیابی به `article_id` و URL اصلی باشند.

تصمیم درباره Duplicate واقعی پس از مشاهده این شواهد گرفته خواهد شد.

</div>

In [20]:
# Code Cell 18

from collections import defaultdict
import hashlib

shortest_records = sorted(
    interim_data,
    key=lambda record: len(record["body_text"].strip()),
)[:15]

print("15 shortest articles:")

for record in shortest_records:
    print(
        f"\narticle_id: {record['article_id']}\n"
        f"title: {record['title']}\n"
        f"body_length: {len(record['body_text'].strip())}\n"
        f"content_origin: {record['content_origin']}\n"
        f"author: {record['author']}\n"
        f"url: {record['canonical_url']}"
    )

title_groups = defaultdict(list)

for record in interim_data:
    normalized_title = record["title"].strip()
    title_groups[normalized_title].append(record)

duplicate_title_groups = {
    title: records
    for title, records in title_groups.items()
    if len(records) > 1
}

print("\n\nDuplicate title groups:", len(duplicate_title_groups))

for title, records in sorted(
    duplicate_title_groups.items(),
    key=lambda item: (-len(item[1]), item[0]),
):
    print(f"\nTitle: {title!r}")
    print("Records:", len(records))

    for record in records:
        print(
            f"  - {record['article_id']} | "
            f"body_length={len(record['body_text'].strip())} | "
            f"{record['canonical_url']}"
        )

body_groups = defaultdict(list)

for record in interim_data:
    normalized_body = record["body_text"].strip()
    body_hash = hashlib.sha256(
        normalized_body.encode("utf-8")
    ).hexdigest()

    body_groups[body_hash].append(record)

duplicate_body_groups = {
    body_hash: records
    for body_hash, records in body_groups.items()
    if len(records) > 1
}

print("\n\nExact duplicate body groups:", len(duplicate_body_groups))

for body_hash, records in duplicate_body_groups.items():
    print(f"\nBody SHA-256: {body_hash}")
    print("Records:", len(records))

    for record in records:
        print(
            f"  - article_id={record['article_id']} | "
            f"title={record['title']!r} | "
            f"author={record['author']!r} | "
            f"origin={record['content_origin']!r} | "
            f"url={record['canonical_url']}"
        )

assert len(shortest_records) == 15

print("\nTEXT QUALITY ANOMALY INSPECTION GATE: PASS")

15 shortest articles:

article_id: 00D9A87F
title: چطور شغل‌مان را توسعه دهیم؟
body_length: 224
content_origin: primary_author
author: دکتر کمیل رودی
url: https://www.fintelligence.ir/Blog/Detail/00D9A87F

article_id: 7E223060
title: ماجرای صندوق سهام دولتی
body_length: 227
content_origin: supervised
author: صاحب‌نظران
url: https://www.fintelligence.ir/Blog/Detail/7E223060

article_id: BF2CFA69
title: کار داوطلبانه، شروعی برای کاریابی و کسب درآمد
body_length: 262
content_origin: primary_author
author: دکتر کمیل رودی
url: https://www.fintelligence.ir/Blog/Detail/BF2CFA69

article_id: 2E656F0E
title: سه دلیل برای سرمایه‌گذاری و پنج گزینه برای انتخاب
body_length: 300
content_origin: primary_author
author: دکتر کمیل رودی
url: https://www.fintelligence.ir/Blog/Detail/2E656F0E

article_id: C203F9BA
title: چرا ریسک‌پذیر نیستیم؟ با نگاهی به صندوق‌های سرمایه‌گذاری
body_length: 310
content_origin: primary_author
author: دکتر کمیل رودی
url: https://www.fintelligence.ir/Blog/Detail/C203F9BA

artic

<div style="direction: rtl; white-space: normal; line-height: 1;">

### تفسیر موارد مشکوک کیفیت متن و بررسی Duplicate واقعی

بررسی انسانی موارد گزارش‌شده نشان می‌دهد کوتاه بودن متن‌ها در وضعیت فعلی به‌تنهایی نشانه خرابی Extraction نیست.

کوتاه‌ترین مقاله 224 کاراکتر دارد و هیچ مقاله‌ای با متن خالی یا کمتر از 100 کاراکتر وجود ندارد. همچنین بخشی از رکوردهای کوتاه مربوط به مطالب کوتاه، معرفی‌ها یا صفحات مرتبط با Podcast هستند.

بنابراین در این مرحله هیچ‌یک از مقالات کوتاه صرفاً بر اساس طول متن Reject نمی‌شوند.

بررسی عنوان‌های تکراری نیز نشان داد 26 Duplicate عنوان در واقع در سه مجموعه محتوایی قرار دارند:

- `کلاهبرداری‌های مالی`
- `مروری بر پژوهش‌های معتبر سواد مالی در جهان`
- `مروری بر پژوهش‌های معتبر سواد مالی در ایران`

این رکوردها دارای `article_id`های متفاوت و طول متن‌های متفاوت هستند؛ بنابراین تکرار عنوان به‌تنهایی Duplicate Document محسوب نمی‌شود و می‌تواند ناشی از انتشار مجموعه‌ای یا سریالی مطالب باشد.

اما یک مورد مهم نیاز به بررسی عمیق‌تر دارد.

دو رکورد زیر دارای `body_text` دقیقاً یکسان هستند:

- `70FB32F5` — معرفی مجموعۀ 6جلدی داستان‌های سواد مالی
- `7D25189D` — معرفی کتاب «من پول هستم»

هر دو توسط `الهام میرمحمدی` نوشته شده و در گروه `supervised` قرار دارند، اما عنوان و `article_id` متفاوت دارند.

این وضعیت می‌تواند یکی از چند حالت باشد:

- دو صفحه واقعی Website محتوای متنی یکسان داشته باشند.
- Extraction در یکی از صفحات متن اشتباهی استخراج کرده باشد.
- یکی از صفحات Duplicate محتوایی دیگری باشد.
- محتوای مشترک عمداً در دو URL متفاوت منتشر شده باشد.

در این مرحله نباید یکی از رکوردها حذف یا Duplicate اعلام شود تا Raw Source بررسی شود.

برای تشخیص علت، دو رکورد از سطح Interim به Raw برگردانده می‌شوند و موارد زیر مقایسه خواهند شد:

- Hash واقعی Raw HTML
- `raw_content_hash`
- اندازه Raw HTML
- Metadata موجود در `source.json`
- عنوان استخراج‌شده
- متن استخراج‌شده
- ابتدای متن برای بررسی انسانی
- یکسان یا متفاوت بودن Raw HTML دو صفحه

**معیار تصمیم:**

اگر Raw HTML دو صفحه نیز یکسان باشد، احتمال Duplicate واقعی در Source بسیار بالا است.

اگر Raw HTML متفاوت ولی `body_text` یکسان باشد، باید بررسی کنیم آیا تفاوت فقط مربوط به عنوان و اجزای صفحه است یا Extraction محتوای یکسانی از دو صفحه متفاوت تولید کرده است.

در این مرحله هنوز هیچ رکوردی حذف یا اصلاح نمی‌شود.

</div>

In [21]:
# Code Cell 19

duplicate_body_article_ids = [
    "70FB32F5",
    "7D25189D",
]

duplicate_body_inspection = {}

for article_id in duplicate_body_article_ids:
    interim_record = interim_by_article_id[article_id]

    article_directory = raw_website_root / article_id
    html_path = article_directory / "article.html"
    source_json_path = article_directory / "source.json"

    html_bytes = html_path.read_bytes()
    html_sha256 = hashlib.sha256(html_bytes).hexdigest()

    with source_json_path.open("r", encoding="utf-8") as file:
        source_metadata = json.load(file)

    body_text = interim_record["body_text"].strip()
    body_sha256 = hashlib.sha256(
        body_text.encode("utf-8")
    ).hexdigest()

    duplicate_body_inspection[article_id] = {
        "title": interim_record["title"],
        "author": interim_record["author"],
        "content_origin": interim_record["content_origin"],
        "canonical_url": interim_record["canonical_url"],
        "body_length": len(body_text),
        "body_sha256": body_sha256,
        "body_preview": body_text[:700],
        "raw_html_size": len(html_bytes),
        "raw_html_sha256": html_sha256,
        "stored_raw_hash": source_metadata["raw_content_hash"],
        "source_metadata": source_metadata,
    }

for article_id, info in duplicate_body_inspection.items():
    print("\n" + "=" * 80)
    print("Article ID:", article_id)
    print("Title:", info["title"])
    print("Author:", info["author"])
    print("Content origin:", info["content_origin"])
    print("Canonical URL:", info["canonical_url"])
    print("Body length:", info["body_length"])
    print("Body SHA-256:", info["body_sha256"])
    print("Raw HTML size:", info["raw_html_size"])
    print("Raw HTML SHA-256:", info["raw_html_sha256"])
    print("Stored Raw Hash:", info["stored_raw_hash"])
    print("\nBody preview:")
    print(info["body_preview"])

first_id, second_id = duplicate_body_article_ids

first = duplicate_body_inspection[first_id]
second = duplicate_body_inspection[second_id]

print("\n" + "=" * 80)
print("PAIR COMPARISON")
print("Same title:", first["title"] == second["title"])
print("Same body hash:", first["body_sha256"] == second["body_sha256"])
print("Same Raw HTML hash:", first["raw_html_sha256"] == second["raw_html_sha256"])
print("Same Raw HTML size:", first["raw_html_size"] == second["raw_html_size"])
print(
    "Each Raw hash matches source.json:",
    first["raw_html_sha256"] == first["stored_raw_hash"]
    and second["raw_html_sha256"] == second["stored_raw_hash"],
)

assert first["body_sha256"] == second["body_sha256"]
assert first["raw_html_sha256"] == first["stored_raw_hash"]
assert second["raw_html_sha256"] == second["stored_raw_hash"]

print("\nEXACT BODY DUPLICATE ROOT-CAUSE INSPECTION GATE: PASS")


Article ID: 70FB32F5
Title: معرفی مجموعۀ 6جلدی داستان‌های سواد مالی
Author: الهام میرمحمدی
Content origin: supervised
Canonical URL: https://www.fintelligence.ir/Blog/Detail/70FB32F5
Body length: 1035
Body SHA-256: 04c4b24bf756760e5ed2f609aa449906085512e52d9b447b2e3205a5e5b97886
Raw HTML size: 71585
Raw HTML SHA-256: f2a2964fc77b987d25b4386ef912a069e5e82d3dda12fed4aa67eeafe6a1d8e2
Stored Raw Hash: f2a2964fc77b987d25b4386ef912a069e5e82d3dda12fed4aa67eeafe6a1d8e2

Body preview:
سواد مالی مجموعه‌ای از دانش‌ها، مهارت‌ها، گرایش‌ها و رفتارهای مالی است که از یک سو به افراد کمک می‌کند سررشتۀ امور مالی‌شان را در دست بگیرند و از سوی دیگر، رشد و شکوفایی اقتصاد جامعه را به دنبال دارد. سواد مالی در شش سرفصل خرج و پس‌انداز، اعتبار و بدهی، کاریابی و درآمد، سرمایه‌گذاری، مدیریت ریسک و بیمه و تصمیم‌گیری مالی به افراد می‌آموزد چگونه امور مالی‌شان را به درستی مدیریت کنند، آرامش و آسایش زندگی را تجربه کرده و امنیت و آزادی مالی را برای خود فراهم کنند.
کارشناسان این حوزه همواره بر آموزش مفاهیم سواد مالی به

<div style="direction: rtl; white-space: normal; line-height: 1;">

### سیاست حفظ محتوای کوتاه و ثبت Quality Flagها

بررسی عمیق Duplicate متنی نشان داد دو رکورد دارای `body_text` کاملاً یکسان هستند، اما Raw HTML آن‌ها یکسان نیست.

دو رکورد دارای:

- `article_id` متفاوت
- عنوان متفاوت
- URL متفاوت
- Raw HTML متفاوت
- Hash خام متفاوت

هستند.

بنابراین این دو رکورد دو Source واقعی و مستقل در Website محسوب می‌شوند و در مرحله Website Acceptance حذف نخواهند شد.

با این حال، چون متن استخراج‌شده آن‌ها کاملاً یکسان است، این رابطه باید به‌عنوان `exact_body_duplicate` ثبت شود تا در مرحله Deduplication و Retrieval از افزایش مصنوعی وزن یک محتوای تکراری جلوگیری شود.

### سیاست مقالات کوتاه

طول کم `body_text` به‌تنهایی معیار Reject شدن یک رکورد نیست.

بخشی از صفحات Website ممکن است مربوط به محتوای چندرسانه‌ای مانند ویدیو یا پادکست باشند و صفحه Website فقط توضیح، معرفی یا خلاصه‌ای از محتوای اصلی را نگهداری کند.

همین متن کوتاه نیز بخشی از Source معتبر Website است و باید حفظ شود.

در مراحل مربوط به Transcript، در صورت وجود متن کامل محتوای صوتی یا ویدیویی، آن محتوا به‌عنوان Source مستقل وارد Knowledge Base خواهد شد.

بنابراین در این پروژه:

`Short Content ≠ Invalid Content`

و:

`Exact Text Duplicate ≠ Automatic Source Deletion`

در مرحله فعلی به‌جای حذف داده، برای هر رکورد Quality Flag ساخته می‌شود.

Flagهای مورد بررسی شامل موارد زیر هستند:

- متن کمتر از 300 کاراکتر
- متن کمتر از 500 کاراکتر
- عنوان تکراری
- متن دقیقاً تکراری
- وجود نشانه صریح محتوای چندرسانه‌ای در عنوان

نشانه چندرسانه‌ای فقط یک Indicator تشخیصی است و به‌تنهایی نوع واقعی محتوا را اثبات نمی‌کند.

**معیار PASS:**

هر 529 رکورد باید در مجموعه Retained باقی بمانند و Quality Flagها بدون حذف یا تغییر Source اصلی محاسبه شوند.

</div>

In [22]:
# Code Cell 20

media_title_keywords = [
    "پادکست",
    "ویدیو",
    "ویدئو",
    "فیلم",
]

duplicate_title_set = set(duplicate_title_groups.keys())
duplicate_body_article_id_set = {
    record["article_id"]
    for records in duplicate_body_groups.values()
    for record in records
}

website_quality_flags = []

for record in interim_data:
    article_id = record["article_id"]
    title = record["title"].strip()
    body_text = record["body_text"].strip()
    body_length = len(body_text)

    media_title_indicator = any(
        keyword in title
        for keyword in media_title_keywords
    )

    flags = {
        "article_id": article_id,
        "content_origin": record["content_origin"],
        "author": record["author"],
        "body_length": body_length,
        "short_under_300": body_length < 300,
        "short_under_500": body_length < 500,
        "duplicate_title": title in duplicate_title_set,
        "exact_body_duplicate": article_id in duplicate_body_article_id_set,
        "media_title_indicator": media_title_indicator,
        "acceptance_action": "retain",
    }

    website_quality_flags.append(flags)

quality_flag_counts = {
    "short_under_300": sum(
        item["short_under_300"]
        for item in website_quality_flags
    ),
    "short_under_500": sum(
        item["short_under_500"]
        for item in website_quality_flags
    ),
    "duplicate_title": sum(
        item["duplicate_title"]
        for item in website_quality_flags
    ),
    "exact_body_duplicate": sum(
        item["exact_body_duplicate"]
        for item in website_quality_flags
    ),
    "media_title_indicator": sum(
        item["media_title_indicator"]
        for item in website_quality_flags
    ),
}

media_by_origin = Counter(
    item["content_origin"]
    for item in website_quality_flags
    if item["media_title_indicator"]
)

retained_count = sum(
    item["acceptance_action"] == "retain"
    for item in website_quality_flags
)

print("Website records evaluated:", len(website_quality_flags))
print("Records retained:", retained_count)

print("\nQuality flags:")
for flag_name, count in quality_flag_counts.items():
    print(f"  {flag_name}: {count}")

print("\nMedia-title indicators by content origin:")
for origin, count in media_by_origin.items():
    print(f"  {origin}: {count}")

print("\nExact-body duplicate records:")
for item in website_quality_flags:
    if item["exact_body_duplicate"]:
        print(
            f"  {item['article_id']} | "
            f"{item['author']} | "
            f"{item['content_origin']}"
        )

assert len(website_quality_flags) == 529
assert retained_count == 529

print("\nWEBSITE CONTENT RETENTION POLICY GATE: PASS")

Website records evaluated: 529
Records retained: 529

Quality flags:
  short_under_300: 3
  short_under_500: 13
  duplicate_title: 29
  exact_body_duplicate: 2
  media_title_indicator: 24

Media-title indicators by content origin:
  supervised: 23
  primary_author: 1

Exact-body duplicate records:
  70FB32F5 | الهام میرمحمدی | supervised
  7D25189D | الهام میرمحمدی | supervised

WEBSITE CONTENT RETENTION POLICY GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### نتیجه سیاست حفظ محتوا و بررسی Artifact پردازش‌شده Website

سیاست حفظ محتوا با موفقیت روی تمام 529 رکورد اعمال شد و هیچ رکوردی صرفاً به دلیل کوتاه بودن متن، عنوان تکراری یا وجود متن مشابه حذف نشد.

نتیجه فعلی:

- رکوردهای بررسی‌شده: 529
- رکوردهای حفظ‌شده: 529
- متن کمتر از 300 کاراکتر: 3 رکورد
- متن کمتر از 500 کاراکتر: 13 رکورد
- رکوردهای دارای عنوان تکراری: 29 رکورد
- رکوردهای دارای متن دقیقاً تکراری: 2 رکورد
- رکوردهای دارای نشانه چندرسانه‌ای در عنوان: 24 رکورد

دو رکورد دارای متن کاملاً یکسان همچنان به‌عنوان دو Source مستقل حفظ شده‌اند، اما با Flag مناسب مشخص شده‌اند تا در مرحله Deduplication و Retrieval کنترل شوند.

یک نکته درباره عدد عنوان‌های تکراری وجود دارد.

در بررسی قبلی مقدار `Duplicate titles = 26` گزارش شده بود. آن عدد تعداد تکرارهای اضافه نسبت به عنوان‌های Unique بود.

در این مرحله مقدار `duplicate_title = 29` تعداد کل رکوردهایی است که عضو یکی از گروه‌های دارای عنوان تکراری هستند.

بنابراین این دو عدد متناقض نیستند و دو معیار متفاوت را نشان می‌دهند.

### نکته مهم درباره صفحات ویدیو و پادکست

از روی عنوان فقط 24 رکورد دارای واژه‌هایی مانند پادکست یا ویدیو تشخیص داده شدند:

- 23 رکورد `supervised`
- 1 رکورد `primary_author`

این عدد به معنی وجود فقط یک محتوای ویدیویی برای دکتر کمیل رودی نیست.

این Check فقط یک Indicator ساده بر اساس متن عنوان است و ممکن است بسیاری از صفحات ویدیویی یا چندرسانه‌ای واژه «ویدیو» یا «پادکست» را در عنوان خود نداشته باشند.

بنابراین تعداد واقعی محتوای چندرسانه‌ای در این Notebook از روی این Indicator تعیین نمی‌شود.

صفحات کوتاه Website همچنان حفظ می‌شوند و اگر نسخه کامل آن‌ها در مجموعه Transcript وجود داشته باشد، در Notebookهای Transcript به‌عنوان Source مستقل وارد سیستم خواهند شد.

### مرحله بعد

تا اینجا زنجیره زیر بررسی شده است:

`Manifest → Raw → Interim`

Pipeline موجود یک Artifact پردازش‌شده دیگر نیز دارد:

`data/phase2/processed/website_documents_v1.json`

اکنون باید ساختار واقعی این فایل را بررسی کنیم.

در این مرحله هنوز فرض نمی‌کنیم که Processed Artifact دقیقاً 529 رکورد دارد یا چه Schemaای دارد.

ابتدا موارد زیر کشف خواهند شد:

- نوع ساختار JSON
- تعداد رکوردها
- تمام فیلدهای موجود
- Coverage هر فیلد
- نوع داده هر فیلد

بعد از شناخت Schema واقعی، ارتباط `Interim → Processed` را جداگانه اعتبارسنجی خواهیم کرد.

**معیار PASS:**

فایل Processed وجود داشته باشد، قابل خواندن باشد و ساختار آن بدون خطا قابل تحلیل باشد.

</div>

In [23]:
# Code Cell 21

processed_path = (
    project_root
    / "data"
    / "phase2"
    / "processed"
    / "website_documents_v1.json"
)

assert processed_path.is_file(), (
    f"Processed artifact not found: {processed_path}"
)

with processed_path.open("r", encoding="utf-8") as file:
    processed_data = json.load(file)

print("Processed artifact:", processed_path.relative_to(project_root))
print("Top-level type:", type(processed_data).__name__)

if isinstance(processed_data, list):
    print("Record count:", len(processed_data))

    assert all(
        isinstance(record, dict)
        for record in processed_data
    ), "One or more processed records are not JSON objects."

    processed_field_presence = Counter()
    processed_field_types = defaultdict(Counter)

    for record in processed_data:
        for field_name, value in record.items():
            processed_field_presence[field_name] += 1
            processed_field_types[field_name][type(value).__name__] += 1

    print("\nField coverage:")

    for field_name in sorted(processed_field_presence):
        count = processed_field_presence[field_name]
        percentage = count / len(processed_data) * 100

        type_summary = ", ".join(
            f"{type_name}:{type_count}"
            for type_name, type_count
            in processed_field_types[field_name].items()
        )

        print(
            f"{field_name:<30} "
            f"{count:>3}/{len(processed_data)} "
            f"({percentage:6.2f}%) "
            f"types=[{type_summary}]"
        )

elif isinstance(processed_data, dict):
    print("Top-level keys:", sorted(processed_data.keys()))

else:
    raise TypeError(
        f"Unsupported processed JSON type: "
        f"{type(processed_data).__name__}"
    )

print("\nPROCESSED ARTIFACT DISCOVERY GATE: PASS")

Processed artifact: data/phase2/processed/website_documents_v1.json
Top-level type: list
Record count: 529

Field coverage:
author                         529/529 (100.00%) types=[str:529]
author_id                      529/529 (100.00%) types=[str:529]
canonical_url                  529/529 (100.00%) types=[str:529]
content                        529/529 (100.00%) types=[str:529]
content_origin                 529/529 (100.00%) types=[str:529]
document_id                    529/529 (100.00%) types=[str:529]
knowledge_owner                529/529 (100.00%) types=[str:529]
metadata                       529/529 (100.00%) types=[dict:529]
raw_content_hash               529/529 (100.00%) types=[str:529]
source_id                      529/529 (100.00%) types=[str:529]
source_type                    529/529 (100.00%) types=[str:529]
title                          529/529 (100.00%) types=[str:529]
url                            529/529 (100.00%) types=[str:529]

PROCESSED ARTIFACT DISCOVERY 

<div style="direction: rtl; white-space: normal; line-height: 1;">

### اعتبارسنجی Lineage میان Interim و Processed

در اجرای اکتشافی اولیه این مرحله، مقایسه میان Interim و Processed نشان داد تقریباً تمام فیلدهای هویتی و محتوایی بدون ناسازگاری منتقل شده‌اند.

با این حال، در فیلد `raw_content_hash` برای 3 رکورد اختلاف مشاهده شد.

این اختلاف باعث شد Gate اولیه این مرحله `FAIL` شود و بررسی Root Cause در مرحله بعد انجام شود.

هدف نسخه نهایی این Code Cell این است که همان Validation را بدون ایجاد Traceback اجرا کند و وضعیت واقعی را به‌صورت صریح گزارش دهد.

اگر ناسازگاری وجود داشته باشد:

- Notebook متوقف نمی‌شود.
- وضعیت Gate به‌صورت `FAIL` ثبت می‌شود.
- شناسه رکوردهای مشکل‌دار برای Root Cause Analysis به مرحله بعد منتقل می‌شوند.

این روش باعث می‌شود Failure واقعی در مستندات حفظ شود، اما اجرای نهایی Notebook با Error فنی متوقف نشود.

</div>

In [27]:
# Code Cell 22

interim_by_source_id = {
    record["source_id"]: record
    for record in interim_data
}

processed_by_source_id = {
    record["source_id"]: record
    for record in processed_data
}

interim_source_ids = set(interim_by_source_id)
processed_source_ids = set(processed_by_source_id)

missing_in_processed = sorted(
    interim_source_ids - processed_source_ids
)

extra_in_processed = sorted(
    processed_source_ids - interim_source_ids
)

document_ids = [
    record["document_id"]
    for record in processed_data
]

empty_document_ids = [
    record["source_id"]
    for record in processed_data
    if not isinstance(record.get("document_id"), str)
    or not record["document_id"].strip()
]

duplicate_document_id_count = (
    len(document_ids)
    - len(set(document_ids))
)

comparison_fields = [
    "title",
    "author",
    "author_id",
    "content_origin",
    "knowledge_owner",
    "url",
    "canonical_url",
    "raw_content_hash",
]

field_mismatches = {
    field_name: []
    for field_name in comparison_fields
}

content_mismatches = []

for source_id in sorted(interim_source_ids & processed_source_ids):
    interim_record = interim_by_source_id[source_id]
    processed_record = processed_by_source_id[source_id]

    for field_name in comparison_fields:
        if interim_record[field_name] != processed_record[field_name]:
            field_mismatches[field_name].append(source_id)

    if interim_record["body_text"] != processed_record["content"]:
        content_mismatches.append(source_id)

document_id_relationships = Counter()

for record in processed_data:
    document_id = record["document_id"]
    source_id = record["source_id"]

    if document_id == source_id:
        relationship = "document_id == source_id"
    elif document_id == f"document:{source_id}":
        relationship = "document:<source_id>"
    else:
        relationship = "other"

    document_id_relationships[relationship] += 1

metadata_field_presence = Counter()
metadata_field_types = defaultdict(Counter)

for record in processed_data:
    metadata = record["metadata"]

    if not isinstance(metadata, dict):
        continue

    for field_name, value in metadata.items():
        metadata_field_presence[field_name] += 1
        metadata_field_types[field_name][type(value).__name__] += 1

print("Missing in Processed:", len(missing_in_processed))
print("Extra in Processed:", len(extra_in_processed))

print("\nDocument IDs:")
print("  Total:", len(document_ids))
print("  Unique:", len(set(document_ids)))
print("  Empty:", len(empty_document_ids))
print("  Duplicates:", duplicate_document_id_count)

print("\nDocument ID relationship:")
for relationship, count in document_id_relationships.most_common():
    print(f"  {relationship}: {count}")

print("\nField mismatches:")
for field_name, mismatches in field_mismatches.items():
    print(f"  {field_name}: {len(mismatches)}")

print("  content/body_text:", len(content_mismatches))

print("\nProcessed metadata schema:")
for field_name in sorted(metadata_field_presence):
    count = metadata_field_presence[field_name]
    percentage = count / len(processed_data) * 100

    type_summary = ", ".join(
        f"{type_name}:{type_count}"
        for type_name, type_count
        in metadata_field_types[field_name].items()
    )

    print(
        f"  {field_name:<25} "
        f"{count:>3}/{len(processed_data)} "
        f"({percentage:6.2f}%) "
        f"types=[{type_summary}]"
    )

critical_non_hash_failures = (
    len(missing_in_processed)
    + len(extra_in_processed)
    + len(empty_document_ids)
    + duplicate_document_id_count
    + len(content_mismatches)
)

for field_name, mismatches in field_mismatches.items():
    if field_name != "raw_content_hash":
        critical_non_hash_failures += len(mismatches)

raw_hash_mismatch_count = len(
    field_mismatches["raw_content_hash"]
)

if critical_non_hash_failures == 0 and raw_hash_mismatch_count == 0:
    processed_lineage_gate_status = "PASS"
elif critical_non_hash_failures == 0 and raw_hash_mismatch_count > 0:
    processed_lineage_gate_status = (
        f"FAIL - {raw_hash_mismatch_count} RAW HASH MISMATCHES"
    )
else:
    processed_lineage_gate_status = (
        f"FAIL - {critical_non_hash_failures} CRITICAL MISMATCHES"
    )

print(
    "\nINTERIM TO PROCESSED LINEAGE GATE:",
    processed_lineage_gate_status,
)

Missing in Processed: 0
Extra in Processed: 0

Document IDs:
  Total: 529
  Unique: 529
  Empty: 0
  Duplicates: 0

Document ID relationship:
  document_id == source_id: 529

Field mismatches:
  title: 0
  author: 0
  author_id: 0
  content_origin: 0
  knowledge_owner: 0
  url: 0
  canonical_url: 0
  raw_content_hash: 3
  content/body_text: 0

Processed metadata schema:
  article_id                529/529 (100.00%) types=[str:529]
  study_time_minutes        529/529 (100.00%) types=[int:529]
  tags                      529/529 (100.00%) types=[list:529]
  view_count                529/529 (100.00%) types=[int:529]

INTERIM TO PROCESSED LINEAGE GATE: FAIL - 3 RAW HASH MISMATCHES


<div style="direction: rtl; white-space: normal; line-height: 1;">

### بررسی علت Failure در Lineage میان Interim و Processed

این مرحله **PASS نشد** و فعلاً نباید از آن عبور کنیم.

بخش عمده ارتباط میان Interim و Processed سالم است:

- رکورد گم‌شده: صفر
- رکورد اضافه: صفر
- `document_id` خالی: صفر
- `document_id` تکراری: صفر
- هر 529 `document_id` دقیقاً برابر `source_id` است.
- عنوان‌ها بدون تغییر منتقل شده‌اند.
- نویسنده‌ها بدون تغییر منتقل شده‌اند.
- `author_id` بدون تغییر منتقل شده است.
- `content_origin` بدون تغییر منتقل شده است.
- `knowledge_owner` بدون تغییر منتقل شده است.
- URLها بدون تغییر منتقل شده‌اند.
- متن `body_text` نیز در هر 529 رکورد دقیقاً به `content` منتقل شده است.

همچنین مشخص شد اطلاعات تکمیلی Interim در Processed داخل `metadata` نگهداری می‌شوند:

- `article_id`
- `study_time_minutes`
- `tags`
- `view_count`

اما یک ناسازگاری مهم مشاهده شد:

**`raw_content_hash` در 3 رکورد میان Interim و Processed متفاوت است.**

چون `raw_content_hash` بخشی از Lineage و Integrity منبع خام است، این اختلاف را نباید نادیده بگیریم یا Assertion را حذف کنیم.

در مرحله قبل ثابت کردیم Hashهای موجود در Raw Metadata با فایل‌های واقعی `article.html` مطابقت دارند. بنابراین اکنون برای همین سه رکورد باید مشخص کنیم کدام لایه مقدار صحیح را نگهداری می‌کند:

`Raw HTML → source.json → Interim → Processed`

برای هر سه مورد موارد زیر کنار هم مقایسه خواهند شد:

- `article_id`
- `source_id`
- عنوان
- Hash واقعی فایل `article.html`
- Hash ذخیره‌شده در `source.json`
- Hash موجود در Interim
- Hash موجود در Processed
- اندازه Raw HTML
- این‌که کدام Hashها با Source واقعی برابر هستند

**معیار تصمیم:**

اگر `source.json` و Interim با Hash واقعی فایل خام برابر باشند ولی Processed متفاوت باشد، مشکل در تولید یا ذخیره Processed Artifact است.

اگر Processed با Raw واقعی برابر باشد و Interim متفاوت باشد، مشکل در Interim است.

اگر هر سه متفاوت باشند، Lineage آن رکورد نیازمند بررسی عمیق‌تر خواهد بود.

تا مشخص شدن علت این سه اختلاف:

**`INTERIM TO PROCESSED LINEAGE GATE = FAIL / INVESTIGATION REQUIRED`**

هیچ داده‌ای در این مرحله تغییر یا اصلاح نمی‌شود.

</div>

In [28]:
# Code Cell 23

raw_hash_mismatch_source_ids = field_mismatches["raw_content_hash"]

print("Raw hash mismatch records:", len(raw_hash_mismatch_source_ids))

raw_hash_root_cause_results = []

for source_id in raw_hash_mismatch_source_ids:
    interim_record = interim_by_source_id[source_id]
    processed_record = processed_by_source_id[source_id]

    article_id = interim_record["article_id"]

    raw_html_path = (
        raw_website_root
        / article_id
        / "article.html"
    )

    raw_source_json_path = (
        raw_website_root
        / article_id
        / "source.json"
    )

    raw_html_bytes = raw_html_path.read_bytes()
    actual_raw_hash = hashlib.sha256(raw_html_bytes).hexdigest()

    with raw_source_json_path.open("r", encoding="utf-8") as file:
        raw_source_metadata = json.load(file)

    source_json_hash = raw_source_metadata["raw_content_hash"]
    interim_hash = interim_record["raw_content_hash"]
    processed_hash = processed_record["raw_content_hash"]

    result = {
        "article_id": article_id,
        "source_id": source_id,
        "title": interim_record["title"],
        "actual_raw_hash": actual_raw_hash,
        "source_json_hash": source_json_hash,
        "interim_hash": interim_hash,
        "processed_hash": processed_hash,
        "raw_size_bytes": len(raw_html_bytes),
        "actual_equals_source_json": actual_raw_hash == source_json_hash,
        "actual_equals_interim": actual_raw_hash == interim_hash,
        "actual_equals_processed": actual_raw_hash == processed_hash,
    }

    raw_hash_root_cause_results.append(result)

    print("\n" + "=" * 80)
    print("Article ID:", article_id)
    print("Source ID:", source_id)
    print("Title:", interim_record["title"])
    print("Raw HTML size:", len(raw_html_bytes))

    print("\nHashes:")
    print("  Actual Raw HTML :", actual_raw_hash)
    print("  source.json     :", source_json_hash)
    print("  Interim         :", interim_hash)
    print("  Processed       :", processed_hash)

    print("\nMatches actual Raw HTML:")
    print("  source.json:", actual_raw_hash == source_json_hash)
    print("  Interim:", actual_raw_hash == interim_hash)
    print("  Processed:", actual_raw_hash == processed_hash)

print("\n" + "=" * 80)

root_cause_patterns = Counter(
    (
        item["actual_equals_source_json"],
        item["actual_equals_interim"],
        item["actual_equals_processed"],
    )
    for item in raw_hash_root_cause_results
)

print("Root-cause patterns:")

for pattern, count in root_cause_patterns.items():
    source_match, interim_match, processed_match = pattern

    print(
        f"  source.json={source_match}, "
        f"interim={interim_match}, "
        f"processed={processed_match}: "
        f"{count}"
    )

assert len(raw_hash_root_cause_results) == 3

print("\nRAW HASH MISMATCH ROOT-CAUSE INSPECTION: COMPLETE")

Raw hash mismatch records: 3

Article ID: 008E61CC
Source ID: website:008E61CC
Title: علف دادم شیر گرفتم
Raw HTML size: 72047

Hashes:
  Actual Raw HTML : 2510dbf81e13f55647cb9e89d4e1f2ecb6dc6276696f8e7871e1132ea5c940bd
  source.json     : 2510dbf81e13f55647cb9e89d4e1f2ecb6dc6276696f8e7871e1132ea5c940bd
  Interim         : 2510dbf81e13f55647cb9e89d4e1f2ecb6dc6276696f8e7871e1132ea5c940bd
  Processed       : e474c6ee3f3b4bc5907af28d88d2e47ac8d925597dbe0f4adfa94a0a75e6ec69

Matches actual Raw HTML:
  source.json: True
  Interim: True
  Processed: False

Article ID: 00D061A8
Source ID: website:00D061A8
Title: وقایع‌نگاری‌های سنجش سواد مالی- 10
Raw HTML size: 77105

Hashes:
  Actual Raw HTML : 957e30668de66a77b6bd657de2d34fd28fb8654d2e72b28d10cdc9071464cfb7
  source.json     : 957e30668de66a77b6bd657de2d34fd28fb8654d2e72b28d10cdc9071464cfb7
  Interim         : 957e30668de66a77b6bd657de2d34fd28fb8654d2e72b28d10cdc9071464cfb7
  Processed       : 813d722f6f83bbdae479729ae32910678bae35b76a4a3ae

<div style="direction: rtl; white-space: normal; line-height: 1;">

### نتیجه Root Cause و ساخت نسخه اصلاح‌شده Processed Artifact

بررسی Root Cause دوباره اجرا شد و همان نتیجه قبلی را تأیید کرد.

در هر سه رکورد مشکل‌دار:

- Hash واقعی فایل `article.html` صحیح است.
- مقدار `raw_content_hash` در `source.json` با فایل واقعی برابر است.
- مقدار `raw_content_hash` در Interim نیز با فایل واقعی برابر است.
- فقط Processed Artifact دارای Hash قدیمی یا ناسازگار است.

رکوردهای مشکل‌دار:

- `website:008E61CC`
- `website:00D061A8`
- `website:00D9A87F`

بنابراین مشکل در لایه Processed قرار دارد و Raw Data یا Interim نیاز به اصلاح ندارند.

### سیاست Repair

نسخه موجود زیر به‌عنوان Artifact تاریخی Pipeline قبلی حفظ می‌شود:

`data/phase2/processed/website_documents_v1.json`

این فایل Overwrite نمی‌شود.

نسخه اصلاح‌شده جدید با نام زیر ساخته خواهد شد:

`data/phase2/processed/website_documents_v2.json`

در نسخه جدید فقط فیلد `raw_content_hash` همان سه رکورد اصلاح خواهد شد.

مقدار صحیح از Interim گرفته می‌شود؛ زیرا در مرحله Root Cause ثابت شد مقدار Interim با `source.json` و فایل Raw واقعی برابر است.

هیچ Content، Title، Attribution، Metadata، ID یا فیلد دیگری نباید تغییر کند.

پس از ساخت فایل جدید موارد زیر بررسی خواهند شد:

- تعداد رکوردها همچنان 529 باشد.
- دقیقاً سه `raw_content_hash` تغییر کرده باشد.
- Content هیچ رکوردی تغییر نکرده باشد.
- `document_id`ها تغییر نکرده باشند.
- هیچ فیلد دیگری تغییر نکرده باشد.
- هر 529 Hash در نسخه جدید با Interim سازگار باشند.

این Repair بخشی از Website Acceptance است و نسخه `v1` همچنان به‌عنوان شواهد وضعیت اولیه حفظ خواهد شد.

</div>

In [29]:
# Code Cell 24

from copy import deepcopy

processed_v2_path = (
    project_root
    / "data"
    / "phase2"
    / "processed"
    / "website_documents_v2.json"
)

processed_data_v2 = deepcopy(processed_data)

repair_source_ids = set(
    field_mismatches["raw_content_hash"]
)

changed_source_ids = []

for record in processed_data_v2:
    source_id = record["source_id"]

    if source_id in repair_source_ids:
        correct_hash = interim_by_source_id[source_id]["raw_content_hash"]

        if record["raw_content_hash"] != correct_hash:
            record["raw_content_hash"] = correct_hash
            changed_source_ids.append(source_id)

processed_v2_path.write_text(
    json.dumps(
        processed_data_v2,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

with processed_v2_path.open("r", encoding="utf-8") as file:
    validated_processed_v2 = json.load(file)

processed_v2_by_source_id = {
    record["source_id"]: record
    for record in validated_processed_v2
}

content_changes = []
document_id_changes = []
unexpected_field_changes = []
remaining_hash_mismatches = []

for source_id in sorted(processed_by_source_id):
    old_record = processed_by_source_id[source_id]
    new_record = processed_v2_by_source_id[source_id]
    interim_record = interim_by_source_id[source_id]

    if old_record["content"] != new_record["content"]:
        content_changes.append(source_id)

    if old_record["document_id"] != new_record["document_id"]:
        document_id_changes.append(source_id)

    for field_name in old_record:
        if field_name == "raw_content_hash":
            continue

        if old_record[field_name] != new_record[field_name]:
            unexpected_field_changes.append(
                {
                    "source_id": source_id,
                    "field": field_name,
                }
            )

    if (
        new_record["raw_content_hash"]
        != interim_record["raw_content_hash"]
    ):
        remaining_hash_mismatches.append(source_id)

print(
    "New processed artifact:",
    processed_v2_path.relative_to(project_root),
)

print("Records:", len(validated_processed_v2))

print("\nRepaired records:", len(changed_source_ids))
for source_id in sorted(changed_source_ids):
    print("  -", source_id)

print("\nPost-repair validation:")
print("Content changes:", len(content_changes))
print("Document ID changes:", len(document_id_changes))
print("Unexpected field changes:", len(unexpected_field_changes))
print("Remaining raw hash mismatches:", len(remaining_hash_mismatches))

assert len(validated_processed_v2) == 529
assert set(changed_source_ids) == repair_source_ids
assert len(changed_source_ids) == 3
assert len(content_changes) == 0
assert len(document_id_changes) == 0
assert len(unexpected_field_changes) == 0
assert len(remaining_hash_mismatches) == 0

print("\nPROCESSED V2 CONTROLLED REPAIR GATE: PASS")

New processed artifact: data/phase2/processed/website_documents_v2.json
Records: 529

Repaired records: 3
  - website:008E61CC
  - website:00D061A8
  - website:00D9A87F

Post-repair validation:
Content changes: 0
Document ID changes: 0
Unexpected field changes: 0
Remaining raw hash mismatches: 0

PROCESSED V2 CONTROLLED REPAIR GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### اعتبارسنجی مستقل Processed Artifact نسخه 2

نسخه اصلاح‌شده Processed Artifact با نام زیر ساخته شد:

`data/phase2/processed/website_documents_v2.json`

در مرحله قبل فقط سه مقدار `raw_content_hash` اصلاح شدند و هیچ Content، ID یا Metadata دیگری تغییر نکرد.

اکنون این فایل به‌صورت مستقل از روی Disk دوباره خوانده می‌شود تا مشخص شود نسخه `v2` واقعاً شرایط لازم برای ادامه Pipeline را دارد.

در این Validation موارد زیر بررسی می‌شوند:

- تعداد رکوردها دقیقاً 529 باشد.
- تمام `source_id`ها Unique باشند.
- تمام `document_id`ها Unique و غیرخالی باشند.
- مجموعه `source_id`ها با Interim یکسان باشد.
- `title` با Interim سازگار باشد.
- `author` با Interim سازگار باشد.
- `author_id` با Interim سازگار باشد.
- `content_origin` با Interim سازگار باشد.
- `knowledge_owner` با Interim سازگار باشد.
- `url` و `canonical_url` با Interim سازگار باشند.
- `content` دقیقاً با `body_text` Interim برابر باشد.
- `raw_content_hash` در هر 529 رکورد با Interim برابر باشد.
- Hash ثبت‌شده در Processed V2 در نهایت با Raw HTML واقعی نیز سازگار باشد.

اگر تمام این موارد PASS شوند، نسخه `v2` به‌عنوان Processed Artifact پذیرفته‌شده Website برای ادامه فاز دوم در نظر گرفته خواهد شد.

نسخه `v1` حذف نمی‌شود، اما از این مرحله به بعد نسخه `v2` باید به‌عنوان Candidate فعال مورد استفاده قرار گیرد.

**معیار PASS:**

تمام ناسازگاری‌ها برابر صفر باشند.

</div>

In [30]:
# Code Cell 25

with processed_v2_path.open("r", encoding="utf-8") as file:
    accepted_processed_data = json.load(file)

assert isinstance(accepted_processed_data, list)
assert len(accepted_processed_data) == 529

accepted_processed_by_source_id = {
    record["source_id"]: record
    for record in accepted_processed_data
}

accepted_source_ids = set(
    accepted_processed_by_source_id
)

accepted_document_ids = [
    record["document_id"]
    for record in accepted_processed_data
]

validation_mismatches = {
    "title": [],
    "author": [],
    "author_id": [],
    "content_origin": [],
    "knowledge_owner": [],
    "url": [],
    "canonical_url": [],
    "content": [],
    "raw_content_hash_interim": [],
    "raw_content_hash_actual": [],
}

for source_id in sorted(interim_by_source_id):
    interim_record = interim_by_source_id[source_id]
    processed_record = accepted_processed_by_source_id[source_id]

    if processed_record["title"] != interim_record["title"]:
        validation_mismatches["title"].append(source_id)

    if processed_record["author"] != interim_record["author"]:
        validation_mismatches["author"].append(source_id)

    if processed_record["author_id"] != interim_record["author_id"]:
        validation_mismatches["author_id"].append(source_id)

    if processed_record["content_origin"] != interim_record["content_origin"]:
        validation_mismatches["content_origin"].append(source_id)

    if processed_record["knowledge_owner"] != interim_record["knowledge_owner"]:
        validation_mismatches["knowledge_owner"].append(source_id)

    if processed_record["url"] != interim_record["url"]:
        validation_mismatches["url"].append(source_id)

    if processed_record["canonical_url"] != interim_record["canonical_url"]:
        validation_mismatches["canonical_url"].append(source_id)

    if processed_record["content"] != interim_record["body_text"]:
        validation_mismatches["content"].append(source_id)

    if (
        processed_record["raw_content_hash"]
        != interim_record["raw_content_hash"]
    ):
        validation_mismatches["raw_content_hash_interim"].append(
            source_id
        )

    article_id = processed_record["metadata"]["article_id"]

    raw_html_path = (
        raw_website_root
        / article_id
        / "article.html"
    )

    actual_raw_hash = hashlib.sha256(
        raw_html_path.read_bytes()
    ).hexdigest()

    if processed_record["raw_content_hash"] != actual_raw_hash:
        validation_mismatches["raw_content_hash_actual"].append(
            source_id
        )

print("Processed V2 records:", len(accepted_processed_data))
print("Unique source IDs:", len(accepted_source_ids))
print("Unique document IDs:", len(set(accepted_document_ids)))

print("\nValidation mismatches:")

for check_name, mismatches in validation_mismatches.items():
    print(f"  {check_name}: {len(mismatches)}")

assert accepted_source_ids == set(interim_by_source_id)
assert len(set(accepted_document_ids)) == 529
assert all(
    isinstance(document_id, str) and document_id.strip()
    for document_id in accepted_document_ids
)

for mismatches in validation_mismatches.values():
    assert len(mismatches) == 0

print("\nPROCESSED V2 INDEPENDENT ACCEPTANCE GATE: PASS")

Processed V2 records: 529
Unique source IDs: 529
Unique document IDs: 529

Validation mismatches:
  title: 0
  author: 0
  author_id: 0
  content_origin: 0
  knowledge_owner: 0
  url: 0
  canonical_url: 0
  content: 0
  raw_content_hash_interim: 0
  raw_content_hash_actual: 0

PROCESSED V2 INDEPENDENT ACCEPTANCE GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### پذیرش نهایی Processed V2 و ورود به بررسی Chunk Artifact

نسخه اصلاح‌شده Processed Artifact با موفقیت از روی Disk دوباره خوانده و به‌صورت مستقل اعتبارسنجی شد.

نتیجه نهایی:

- تعداد Documentها: 529
- `source_id` یکتا: 529
- `document_id` یکتا: 529
- اختلاف عنوان: صفر
- اختلاف نویسنده: صفر
- اختلاف `author_id`: صفر
- اختلاف `content_origin`: صفر
- اختلاف `knowledge_owner`: صفر
- اختلاف URL: صفر
- اختلاف Canonical URL: صفر
- اختلاف Content: صفر
- اختلاف Hash با Interim: صفر
- اختلاف Hash با Raw HTML واقعی: صفر

بنابراین:

`website_documents_v2.json`

از این مرحله به بعد **Processed Artifact پذیرفته‌شده Website** محسوب می‌شود.

نسخه `v1` فقط برای حفظ Audit Trail باقی می‌ماند و نباید ورودی مراحل جدید باشد.

### مرحله بعد — بررسی Chunk Artifact موجود

Pipeline قبلی یک Artifact دیگر نیز تولید کرده است:

`data/phase2/chunks/website_chunks_v2.json`

وجود این فایل به معنی پذیرفته‌شدن آن نیست.

به‌خصوص چون Processed Artifact قبلی دارای سه Metadata Hash ناسازگار بود، باید بررسی کنیم Chunkهای موجود:

- از کدام نسخه Processed ساخته شده‌اند.
- چه Schemaای دارند.
- چند Chunk وجود دارد.
- آیا تمام 529 Document در آن‌ها پوشش داده شده‌اند.
- آیا Stable ID و Lineage حفظ شده است.
- آیا Chunkها هنوز به Metadata قدیمی متصل هستند یا خیر.

در این مرحله فقط ساختار واقعی Chunk Artifact کشف می‌شود و هنوز هیچ Chunkی اصلاح یا بازسازی نخواهد شد.

موارد زیر بررسی می‌شوند:

- نوع ساختار JSON
- تعداد Chunkها
- Schema واقعی
- Coverage هر فیلد
- نوع داده هر فیلد

**معیار PASS این مرحله:**

فایل Chunk موجود باشد، قابل خواندن باشد و ساختار آن بدون خطا قابل تحلیل باشد.

پذیرش واقعی Chunkها بعد از بررسی Lineage و کیفیت انجام خواهد شد.

</div>

In [31]:
# Code Cell 26

chunks_path = (
    project_root
    / "data"
    / "phase2"
    / "chunks"
    / "website_chunks_v2.json"
)

assert chunks_path.is_file(), (
    f"Chunk artifact not found: {chunks_path}"
)

with chunks_path.open("r", encoding="utf-8") as file:
    chunks_data = json.load(file)

print("Chunk artifact:", chunks_path.relative_to(project_root))
print("Top-level type:", type(chunks_data).__name__)

if isinstance(chunks_data, list):
    print("Chunk count:", len(chunks_data))

    assert all(
        isinstance(record, dict)
        for record in chunks_data
    ), "One or more chunk records are not JSON objects."

    chunk_field_presence = Counter()
    chunk_field_types = defaultdict(Counter)

    for record in chunks_data:
        for field_name, value in record.items():
            chunk_field_presence[field_name] += 1
            chunk_field_types[field_name][type(value).__name__] += 1

    print("\nField coverage:")

    for field_name in sorted(chunk_field_presence):
        count = chunk_field_presence[field_name]
        percentage = count / len(chunks_data) * 100

        type_summary = ", ".join(
            f"{type_name}:{type_count}"
            for type_name, type_count
            in chunk_field_types[field_name].items()
        )

        print(
            f"{field_name:<30} "
            f"{count:>4}/{len(chunks_data)} "
            f"({percentage:6.2f}%) "
            f"types=[{type_summary}]"
        )

elif isinstance(chunks_data, dict):
    print("Top-level keys:", sorted(chunks_data.keys()))

else:
    raise TypeError(
        f"Unsupported chunk JSON type: "
        f"{type(chunks_data).__name__}"
    )

print("\nCHUNK ARTIFACT DISCOVERY GATE: PASS")

Chunk artifact: data/phase2/chunks/website_chunks_v2.json
Top-level type: list
Chunk count: 1116

Field coverage:
author                         1116/1116 (100.00%) types=[str:1116]
canonical_url                  1116/1116 (100.00%) types=[str:1116]
chunk_count                    1116/1116 (100.00%) types=[int:1116]
chunk_id                       1116/1116 (100.00%) types=[str:1116]
chunk_index                    1116/1116 (100.00%) types=[int:1116]
content                        1116/1116 (100.00%) types=[str:1116]
content_origin                 1116/1116 (100.00%) types=[str:1116]
document_id                    1116/1116 (100.00%) types=[str:1116]
knowledge_owner                1116/1116 (100.00%) types=[str:1116]
metadata                       1116/1116 (100.00%) types=[dict:1116]
raw_content_hash               1116/1116 (100.00%) types=[str:1116]
source_id                      1116/1116 (100.00%) types=[str:1116]
source_type                    1116/1116 (100.00%) types=[str:1116]
t

<div style="direction: rtl; white-space: normal; line-height: 1;">

### بررسی ساختار و Lineage فایل Chunk موجود

فایل `website_chunks_v2.json` شامل 1116 Chunk است و تمام Chunkها دارای Schema یکسان هستند.

این فایل در Pipeline قبلی ساخته شده است، اما هنوز مشخص نیست که از نظر Lineage با Processed Artifact پذیرفته‌شده جدید یعنی:

`website_documents_v2.json`

کاملاً سازگار باشد.

در Notebook 02 هدف ما Benchmark کردن روش Chunking نیست.

انتخاب اندازه Chunk، Overlap، کیفیت Semantic Split و مقایسه روش‌های مختلف Chunking در Notebook مخصوص Chunking Benchmark انجام خواهد شد.

در این مرحله فقط بررسی می‌کنیم که Chunk Artifact موجود از نظر **هویت، پوشش و Lineage** سالم است یا خیر.

موارد زیر بررسی خواهند شد:

- تعداد `chunk_id`های Unique
- وجود Chunk برای تمام 529 Document
- وجود Document اضافه یا ناشناخته
- ارتباط `document_id` و `source_id`
- ترتیب `chunk_index` برای هر Document
- سازگاری `chunk_count` ذخیره‌شده با تعداد واقعی Chunkهای هر Document
- سازگاری Attribution و Metadata اصلی Chunk با Processed V2
- سازگاری `raw_content_hash` هر Chunk با Processed V2
- Schema داخلی فیلد `metadata`

### نکته مهم

چون در Processed V1 سه مقدار `raw_content_hash` اشتباه پیدا کردیم، احتمال دارد Chunk Artifact موجود نیز همان Metadata قدیمی را حمل کرده باشد.

اگر چنین اختلافی پیدا شود، آن را مخفی نمی‌کنیم و Gate به‌صورت `FAIL` گزارش می‌شود، اما Notebook با Traceback متوقف نخواهد شد تا بتوانیم Root Cause را بررسی کنیم.

در این Notebook هنوز Chunkها بازسازی یا Rechunk نمی‌شوند.

**معیار PASS:**

- تمام 529 Document در Chunkها پوشش داده شده باشند.
- Chunk ناشناخته وجود نداشته باشد.
- `chunk_id`ها Unique باشند.
- ترتیب و تعداد Chunkها برای هر Document سازگار باشد.
- Metadata هویتی و Attribution بدون اختلاف منتقل شده باشد.
- `raw_content_hash` با Processed V2 سازگار باشد.

</div>

In [32]:
# Code Cell 27

from collections import defaultdict, Counter

chunk_ids = [
    chunk["chunk_id"]
    for chunk in chunks_data
]

chunk_source_ids = {
    chunk["source_id"]
    for chunk in chunks_data
}

chunk_document_ids = {
    chunk["document_id"]
    for chunk in chunks_data
}

accepted_processed_source_ids = set(
    accepted_processed_by_source_id
)

missing_documents_in_chunks = sorted(
    accepted_processed_source_ids - chunk_source_ids
)

extra_documents_in_chunks = sorted(
    chunk_source_ids - accepted_processed_source_ids
)

chunks_by_source_id = defaultdict(list)

for chunk in chunks_data:
    chunks_by_source_id[chunk["source_id"]].append(chunk)

chunk_structure_anomalies = []
chunk_count_mismatches = []
document_source_id_mismatches = []

for source_id, document_chunks in chunks_by_source_id.items():
    ordered_chunks = sorted(
        document_chunks,
        key=lambda item: item["chunk_index"],
    )

    actual_chunk_count = len(ordered_chunks)

    expected_indexes = list(range(actual_chunk_count))
    actual_indexes = [
        chunk["chunk_index"]
        for chunk in ordered_chunks
    ]

    declared_chunk_counts = {
        chunk["chunk_count"]
        for chunk in ordered_chunks
    }

    if actual_indexes != expected_indexes:
        chunk_structure_anomalies.append(source_id)

    if (
        len(declared_chunk_counts) != 1
        or next(iter(declared_chunk_counts)) != actual_chunk_count
    ):
        chunk_count_mismatches.append(source_id)

    for chunk in ordered_chunks:
        if chunk["document_id"] != chunk["source_id"]:
            document_source_id_mismatches.append(
                chunk["chunk_id"]
            )

lineage_fields = [
    "title",
    "author",
    "content_origin",
    "knowledge_owner",
    "canonical_url",
    "source_type",
    "raw_content_hash",
]

chunk_lineage_mismatches = {
    field_name: []
    for field_name in lineage_fields
}

for chunk in chunks_data:
    source_id = chunk["source_id"]

    if source_id not in accepted_processed_by_source_id:
        continue

    processed_record = accepted_processed_by_source_id[source_id]

    for field_name in lineage_fields:
        if chunk[field_name] != processed_record[field_name]:
            chunk_lineage_mismatches[field_name].append(
                chunk["chunk_id"]
            )

chunk_metadata_field_presence = Counter()
chunk_metadata_field_types = defaultdict(Counter)

for chunk in chunks_data:
    metadata = chunk["metadata"]

    if not isinstance(metadata, dict):
        continue

    for field_name, value in metadata.items():
        chunk_metadata_field_presence[field_name] += 1
        chunk_metadata_field_types[field_name][type(value).__name__] += 1

print("Total chunks:", len(chunks_data))
print("Unique chunk IDs:", len(set(chunk_ids)))
print("Documents represented in chunks:", len(chunk_source_ids))

print("\nDocument coverage:")
print("  Missing documents:", len(missing_documents_in_chunks))
print("  Extra documents:", len(extra_documents_in_chunks))

print("\nChunk structure:")
print("  Chunk index anomalies:", len(chunk_structure_anomalies))
print("  Chunk count mismatches:", len(chunk_count_mismatches))
print(
    "  document_id/source_id mismatches:",
    len(document_source_id_mismatches),
)

print("\nLineage mismatches against Processed V2:")

for field_name, mismatches in chunk_lineage_mismatches.items():
    print(f"  {field_name}: {len(mismatches)}")

print("\nChunk metadata schema:")

for field_name in sorted(chunk_metadata_field_presence):
    count = chunk_metadata_field_presence[field_name]
    percentage = count / len(chunks_data) * 100

    type_summary = ", ".join(
        f"{type_name}:{type_count}"
        for type_name, type_count
        in chunk_metadata_field_types[field_name].items()
    )

    print(
        f"  {field_name:<25} "
        f"{count:>4}/{len(chunks_data)} "
        f"({percentage:6.2f}%) "
        f"types=[{type_summary}]"
    )

structural_failure_count = (
    len(chunk_ids) - len(set(chunk_ids))
    + len(missing_documents_in_chunks)
    + len(extra_documents_in_chunks)
    + len(chunk_structure_anomalies)
    + len(chunk_count_mismatches)
    + len(document_source_id_mismatches)
)

non_hash_lineage_failures = sum(
    len(mismatches)
    for field_name, mismatches in chunk_lineage_mismatches.items()
    if field_name != "raw_content_hash"
)

chunk_raw_hash_mismatches = len(
    chunk_lineage_mismatches["raw_content_hash"]
)

if (
    structural_failure_count == 0
    and non_hash_lineage_failures == 0
    and chunk_raw_hash_mismatches == 0
):
    chunk_lineage_gate_status = "PASS"

elif (
    structural_failure_count == 0
    and non_hash_lineage_failures == 0
    and chunk_raw_hash_mismatches > 0
):
    chunk_lineage_gate_status = (
        f"FAIL - {chunk_raw_hash_mismatches} "
        "RAW HASH MISMATCHED CHUNKS"
    )

else:
    chunk_lineage_gate_status = (
        f"FAIL - structural={structural_failure_count}, "
        f"non_hash_lineage={non_hash_lineage_failures}, "
        f"raw_hash={chunk_raw_hash_mismatches}"
    )

print(
    "\nCHUNK STRUCTURE AND LINEAGE GATE:",
    chunk_lineage_gate_status,
)

Total chunks: 1116
Unique chunk IDs: 1116
Documents represented in chunks: 529

Document coverage:
  Missing documents: 0
  Extra documents: 0

Chunk structure:
  Chunk index anomalies: 529
  Chunk count mismatches: 0
  document_id/source_id mismatches: 0

Lineage mismatches against Processed V2:
  title: 0
  author: 0
  content_origin: 0
  knowledge_owner: 0
  canonical_url: 0
  source_type: 0
  raw_content_hash: 5

Chunk metadata schema:
  article_id                1116/1116 (100.00%) types=[str:1116]
  study_time_minutes        1116/1116 (100.00%) types=[int:1116]
  tags                      1116/1116 (100.00%) types=[list:1116]
  view_count                1116/1116 (100.00%) types=[int:1116]

CHUNK STRUCTURE AND LINEAGE GATE: FAIL - structural=529, non_hash_lineage=0, raw_hash=5


<div style="direction: rtl; white-space: normal; line-height: 1;">

### بررسی علت Failure در Chunk Artifact موجود

در این مرحله هیچ Chunk جدیدی ساخته نشده است.

فایل `website_chunks_v2.json` یک Artifact موجود از Pipeline قبلی است و در Notebook 02 فقط از نظر Acceptance و Lineage بررسی می‌شود.

بررسی اولیه نشان داد:

- تعداد Chunkها: 1116
- تعداد Documentهای پوشش‌داده‌شده: 529
- Document گم‌شده: صفر
- Document اضافه: صفر
- `chunk_id` تکراری: صفر
- اختلاف Metadata غیر از Hash: صفر
- Chunk دارای `raw_content_hash` ناسازگار: 5
- Chunk Index Anomaly: 529 Document

وجود Anomaly برای تمام 529 Document می‌تواند نشان‌دهنده تفاوت Contract شماره‌گذاری باشد؛ برای مثال ممکن است Chunk Indexهای Pipeline قبلی از 1 شروع شده باشند، در حالی که Validation فعلی انتظار شروع از 0 داشت.

اما این موضوع تا مشاهده داده واقعی قطعی نیست.

همچنین پنج Chunk دارای Hash ناسازگار باید به `source_id`های اصلی آن‌ها متصل شوند تا مشخص شود آیا همان سه Document دارای Hash قدیمی در Processed V1 هستند یا مشکل دیگری وجود دارد.

در این مرحله فقط Root Cause بررسی می‌شود و هیچ Rechunking یا تغییر Artifact انجام نخواهد شد.

</div>

In [33]:
# Code Cell 28

chunk_index_patterns = Counter()
chunk_index_samples = []

for source_id, document_chunks in sorted(chunks_by_source_id.items()):
    indexes = sorted(
        chunk["chunk_index"]
        for chunk in document_chunks
    )

    chunk_index_patterns[
        (
            indexes[0],
            indexes[-1],
            len(indexes),
        )
    ] += 1

    if len(chunk_index_samples) < 10:
        chunk_index_samples.append(
            {
                "source_id": source_id,
                "indexes": indexes,
                "declared_chunk_count": document_chunks[0]["chunk_count"],
            }
        )

hash_mismatch_chunks = []

for chunk in chunks_data:
    source_id = chunk["source_id"]

    if source_id not in accepted_processed_by_source_id:
        continue

    processed_hash = accepted_processed_by_source_id[source_id][
        "raw_content_hash"
    ]

    if chunk["raw_content_hash"] != processed_hash:
        hash_mismatch_chunks.append(
            {
                "chunk_id": chunk["chunk_id"],
                "source_id": source_id,
                "chunk_index": chunk["chunk_index"],
                "chunk_hash": chunk["raw_content_hash"],
                "processed_v2_hash": processed_hash,
            }
        )

print("Chunk index pattern samples:")

for sample in chunk_index_samples:
    print(
        f"  {sample['source_id']} | "
        f"indexes={sample['indexes']} | "
        f"declared_count={sample['declared_chunk_count']}"
    )

print("\nIndex start distribution:")

index_start_distribution = Counter(
    min(
        chunk["chunk_index"]
        for chunk in document_chunks
    )
    for document_chunks in chunks_by_source_id.values()
)

for start_index, count in sorted(index_start_distribution.items()):
    print(f"  starts_at={start_index}: {count} documents")

print("\nRaw hash mismatched chunks:", len(hash_mismatch_chunks))

for item in hash_mismatch_chunks:
    print(
        f"  chunk_id={item['chunk_id']} | "
        f"source_id={item['source_id']} | "
        f"chunk_index={item['chunk_index']}"
    )

print(
    "\nUnique source IDs with hash mismatch:",
    len({item["source_id"] for item in hash_mismatch_chunks}),
)

print("\nCHUNK ROOT-CAUSE INSPECTION: COMPLETE")

Chunk index pattern samples:
  website:008E61CC | indexes=[1, 2] | declared_count=2
  website:00D061A8 | indexes=[1, 2] | declared_count=2
  website:00D9A87F | indexes=[1] | declared_count=1
  website:00EDA828 | indexes=[1, 2, 3] | declared_count=3
  website:0152EE71 | indexes=[1, 2] | declared_count=2
  website:01AE96B5 | indexes=[1] | declared_count=1
  website:01D66D39 | indexes=[1, 2] | declared_count=2
  website:021FED66 | indexes=[1, 2] | declared_count=2
  website:02376891 | indexes=[1] | declared_count=1
  website:0248318E | indexes=[1, 2, 3] | declared_count=3

Index start distribution:
  starts_at=1: 529 documents

Raw hash mismatched chunks: 5
  chunk_id=website:008E61CC:chunk:1 | source_id=website:008E61CC | chunk_index=1
  chunk_id=website:008E61CC:chunk:2 | source_id=website:008E61CC | chunk_index=2
  chunk_id=website:00D061A8:chunk:1 | source_id=website:00D061A8 | chunk_index=1
  chunk_id=website:00D061A8:chunk:2 | source_id=website:00D061A8 | chunk_index=2
  chunk_id=we

<div style="direction: rtl; white-space: normal; line-height: 1;">

### اصلاح Contract مربوط به Chunk Index و اعتبارسنجی مجدد Lineage

بررسی Root Cause نشان داد تمام 529 Document از `chunk_index = 1` شروع می‌شوند و شماره‌گذاری Chunkها به‌صورت پیوسته ادامه پیدا می‌کند.

بنابراین Pipeline موجود از **1-based indexing** استفاده کرده است.

در Validation قبلی، انتظار اشتباه برای شروع Index از صفر باعث شد تمام 529 Document به‌عنوان Anomaly گزارش شوند.

این مورد یک مشکل واقعی در داده نبود و باید با Contract صحیح دوباره ارزیابی شود.

در Validation جدید موارد زیر بررسی خواهند شد:

- تمام `chunk_id`ها Unique باشند.
- تمام 529 Document در Chunk Artifact حضور داشته باشند.
- هیچ Document اضافه وجود نداشته باشد.
- `chunk_index` برای هر Document از 1 شروع شود و به‌صورت پیوسته ادامه پیدا کند.
- مقدار `chunk_count` با تعداد واقعی Chunkهای هر Document سازگار باشد.
- `document_id` با `source_id` سازگار باشد.
- Metadata هویتی و Attribution با Processed V2 یکسان باشد.
- `raw_content_hash` هر Chunk با Processed V2 مقایسه شود.

بررسی قبلی همچنین نشان داد 5 Chunk دارای `raw_content_hash` ناسازگار هستند.

این 5 Chunk متعلق به همان 3 Documentی هستند که در Processed V1 دارای Hash قدیمی بودند.

در این مرحله Chunkها تغییر یا بازسازی نمی‌شوند و فقط وضعیت واقعی آن‌ها دوباره با Contract صحیح گزارش می‌شود.

**معیار PASS:**

تمام Validationهای ساختاری و Lineage باید بدون اختلاف باشند.

اگر تنها اختلاف باقی‌مانده مربوط به `raw_content_hash` باشد، Gate به‌صورت `FAIL` گزارش می‌شود تا در مرحله بعد Root Cause و Repair همان Metadata انجام شود.

</div>

In [34]:
# Code Cell 27

chunk_ids = [
    chunk["chunk_id"]
    for chunk in chunks_data
]

chunk_source_ids = {
    chunk["source_id"]
    for chunk in chunks_data
}

accepted_processed_source_ids = set(
    accepted_processed_by_source_id
)

missing_documents_in_chunks = sorted(
    accepted_processed_source_ids - chunk_source_ids
)

extra_documents_in_chunks = sorted(
    chunk_source_ids - accepted_processed_source_ids
)

chunks_by_source_id = defaultdict(list)

for chunk in chunks_data:
    chunks_by_source_id[chunk["source_id"]].append(chunk)

chunk_index_anomalies = []
chunk_count_mismatches = []
document_source_id_mismatches = []

for source_id, document_chunks in chunks_by_source_id.items():
    ordered_chunks = sorted(
        document_chunks,
        key=lambda item: item["chunk_index"],
    )

    actual_chunk_count = len(ordered_chunks)

    expected_indexes = list(
        range(1, actual_chunk_count + 1)
    )

    actual_indexes = [
        chunk["chunk_index"]
        for chunk in ordered_chunks
    ]

    declared_chunk_counts = {
        chunk["chunk_count"]
        for chunk in ordered_chunks
    }

    if actual_indexes != expected_indexes:
        chunk_index_anomalies.append(source_id)

    if (
        len(declared_chunk_counts) != 1
        or next(iter(declared_chunk_counts)) != actual_chunk_count
    ):
        chunk_count_mismatches.append(source_id)

    for chunk in ordered_chunks:
        if chunk["document_id"] != chunk["source_id"]:
            document_source_id_mismatches.append(
                chunk["chunk_id"]
            )

lineage_fields = [
    "title",
    "author",
    "content_origin",
    "knowledge_owner",
    "canonical_url",
    "source_type",
    "raw_content_hash",
]

chunk_lineage_mismatches = {
    field_name: []
    for field_name in lineage_fields
}

for chunk in chunks_data:
    source_id = chunk["source_id"]

    if source_id not in accepted_processed_by_source_id:
        continue

    processed_record = accepted_processed_by_source_id[source_id]

    for field_name in lineage_fields:
        if chunk[field_name] != processed_record[field_name]:
            chunk_lineage_mismatches[field_name].append(
                chunk["chunk_id"]
            )

print("Total chunks:", len(chunks_data))
print("Unique chunk IDs:", len(set(chunk_ids)))
print("Documents represented in chunks:", len(chunk_source_ids))

print("\nDocument coverage:")
print("  Missing documents:", len(missing_documents_in_chunks))
print("  Extra documents:", len(extra_documents_in_chunks))

print("\nChunk structure:")
print("  Chunk index anomalies:", len(chunk_index_anomalies))
print("  Chunk count mismatches:", len(chunk_count_mismatches))
print(
    "  document_id/source_id mismatches:",
    len(document_source_id_mismatches),
)

print("\nLineage mismatches against Processed V2:")

for field_name, mismatches in chunk_lineage_mismatches.items():
    print(f"  {field_name}: {len(mismatches)}")

structural_failure_count = (
    len(chunk_ids) - len(set(chunk_ids))
    + len(missing_documents_in_chunks)
    + len(extra_documents_in_chunks)
    + len(chunk_index_anomalies)
    + len(chunk_count_mismatches)
    + len(document_source_id_mismatches)
)

non_hash_lineage_failures = sum(
    len(mismatches)
    for field_name, mismatches in chunk_lineage_mismatches.items()
    if field_name != "raw_content_hash"
)

chunk_raw_hash_mismatches = len(
    chunk_lineage_mismatches["raw_content_hash"]
)

if (
    structural_failure_count == 0
    and non_hash_lineage_failures == 0
    and chunk_raw_hash_mismatches == 0
):
    chunk_lineage_gate_status = "PASS"

elif (
    structural_failure_count == 0
    and non_hash_lineage_failures == 0
    and chunk_raw_hash_mismatches > 0
):
    chunk_lineage_gate_status = (
        f"FAIL - {chunk_raw_hash_mismatches} "
        "RAW HASH MISMATCHED CHUNKS"
    )

else:
    chunk_lineage_gate_status = (
        f"FAIL - structural={structural_failure_count}, "
        f"non_hash_lineage={non_hash_lineage_failures}, "
        f"raw_hash={chunk_raw_hash_mismatches}"
    )

print(
    "\nCHUNK STRUCTURE AND LINEAGE GATE:",
    chunk_lineage_gate_status,
)

Total chunks: 1116
Unique chunk IDs: 1116
Documents represented in chunks: 529

Document coverage:
  Missing documents: 0
  Extra documents: 0

Chunk structure:
  Chunk index anomalies: 0
  Chunk count mismatches: 0
  document_id/source_id mismatches: 0

Lineage mismatches against Processed V2:
  title: 0
  author: 0
  content_origin: 0
  knowledge_owner: 0
  canonical_url: 0
  source_type: 0
  raw_content_hash: 5

CHUNK STRUCTURE AND LINEAGE GATE: FAIL - 5 RAW HASH MISMATCHED CHUNKS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### تعمیر کنترل‌شده Metadata در Chunk Artifact

اعتبارسنجی مجدد Chunk Artifact با Contract صحیح برای `chunk_index` نشان داد ساختار Chunkها سالم است.

نتیجه فعلی:

- تعداد Chunkها: 1116
- `chunk_id` تکراری: صفر
- تعداد Documentهای پوشش‌داده‌شده: 529
- Document گم‌شده: صفر
- Document اضافه: صفر
- خطای ترتیب `chunk_index`: صفر
- خطای `chunk_count`: صفر
- اختلاف `document_id` و `source_id`: صفر
- اختلاف Attribution و Metadata هویتی: صفر
- اختلاف `raw_content_hash`: 5 Chunk

بررسی Root Cause قبلی نشان داد این 5 Chunk متعلق به همان 3 Documentی هستند که Processed V1 دارای Hash قدیمی بود.

بنابراین مشکل مربوط به Chunk Boundary، متن Chunk یا روش Chunking نیست.

فقط Metadata مربوط به `raw_content_hash` از Processed V1 به این Chunkها منتقل شده است.

در این مرحله هیچ Rechunking انجام نمی‌شود.

نسخه موجود:

`data/phase2/chunks/website_chunks_v2.json`

به‌عنوان Artifact تاریخی حفظ خواهد شد و Overwrite نمی‌شود.

نسخه اصلاح‌شده جدید با نام زیر ساخته می‌شود:

`data/phase2/chunks/website_chunks_v3.json`

در نسخه جدید فقط `raw_content_hash` Chunkهای ناسازگار از Processed V2 پذیرفته‌شده گرفته می‌شود.

هیچ تغییر دیگری در:

- `content`
- `chunk_id`
- `chunk_index`
- `chunk_count`
- `document_id`
- `source_id`
- Attribution
- Metadata

مجاز نیست.

**معیار PASS:**

- تعداد Chunkها همچنان 1116 باشد.
- دقیقاً 5 مقدار `raw_content_hash` اصلاح شود.
- هیچ Content یا فیلد دیگری تغییر نکند.
- پس از Repair، تمام 1116 Chunk با Processed V2 از نظر `raw_content_hash` سازگار باشند.

</div>

In [35]:
# Code Cell 29

from copy import deepcopy

chunks_v3_path = (
    project_root
    / "data"
    / "phase2"
    / "chunks"
    / "website_chunks_v3.json"
)

chunks_data_v3 = deepcopy(chunks_data)

repaired_chunk_ids = []

for chunk in chunks_data_v3:
    source_id = chunk["source_id"]

    expected_hash = accepted_processed_by_source_id[source_id][
        "raw_content_hash"
    ]

    if chunk["raw_content_hash"] != expected_hash:
        chunk["raw_content_hash"] = expected_hash
        repaired_chunk_ids.append(chunk["chunk_id"])

chunks_v3_path.write_text(
    json.dumps(
        chunks_data_v3,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

with chunks_v3_path.open("r", encoding="utf-8") as file:
    validated_chunks_v3 = json.load(file)

assert len(validated_chunks_v3) == len(chunks_data)

original_chunks_by_id = {
    chunk["chunk_id"]: chunk
    for chunk in chunks_data
}

validated_chunks_v3_by_id = {
    chunk["chunk_id"]: chunk
    for chunk in validated_chunks_v3
}

unexpected_changes = []
content_changes = []
remaining_hash_mismatches = []

for chunk_id, new_chunk in validated_chunks_v3_by_id.items():
    old_chunk = original_chunks_by_id[chunk_id]

    if old_chunk["content"] != new_chunk["content"]:
        content_changes.append(chunk_id)

    for field_name in old_chunk:
        if field_name == "raw_content_hash":
            continue

        if old_chunk[field_name] != new_chunk[field_name]:
            unexpected_changes.append(
                {
                    "chunk_id": chunk_id,
                    "field": field_name,
                }
            )

    source_id = new_chunk["source_id"]

    expected_hash = accepted_processed_by_source_id[source_id][
        "raw_content_hash"
    ]

    if new_chunk["raw_content_hash"] != expected_hash:
        remaining_hash_mismatches.append(chunk_id)

print(
    "New chunk artifact:",
    chunks_v3_path.relative_to(project_root),
)

print("Chunks:", len(validated_chunks_v3))

print("\nRepaired raw_content_hash values:", len(repaired_chunk_ids))
for chunk_id in repaired_chunk_ids:
    print("  -", chunk_id)

print("\nPost-repair validation:")
print("Content changes:", len(content_changes))
print("Unexpected field changes:", len(unexpected_changes))
print("Remaining raw hash mismatches:", len(remaining_hash_mismatches))

assert len(repaired_chunk_ids) == 5
assert len(content_changes) == 0
assert len(unexpected_changes) == 0
assert len(remaining_hash_mismatches) == 0

print("\nCHUNKS V3 CONTROLLED METADATA REPAIR GATE: PASS")

New chunk artifact: data/phase2/chunks/website_chunks_v3.json
Chunks: 1116

Repaired raw_content_hash values: 5
  - website:008E61CC:chunk:1
  - website:008E61CC:chunk:2
  - website:00D061A8:chunk:1
  - website:00D061A8:chunk:2
  - website:00D9A87F:chunk:1

Post-repair validation:
Content changes: 0
Unexpected field changes: 0
Remaining raw hash mismatches: 0

CHUNKS V3 CONTROLLED METADATA REPAIR GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### اعتبارسنجی مستقل Chunk Artifact نسخه 3

نسخه اصلاح‌شده Chunk Artifact با موفقیت ساخته شد:

`data/phase2/chunks/website_chunks_v3.json`

در مرحله قبل فقط 5 مقدار `raw_content_hash` اصلاح شدند و هیچ Content، شناسه، ترتیب Chunk یا Metadata دیگری تغییر نکرد.

اکنون این فایل به‌صورت مستقل دوباره از روی Disk خوانده می‌شود تا مشخص شود نسخه `v3` واقعاً شرایط لازم برای پذیرش فنی را دارد.

موارد زیر بررسی خواهند شد:

- تعداد Chunkها دقیقاً 1116 باشد.
- تمام `chunk_id`ها Unique باشند.
- هر 529 Document حداقل یک Chunk داشته باشد.
- هیچ Document اضافه یا گم‌شده وجود نداشته باشد.
- `chunk_index` برای هر Document از 1 شروع شود و پیوسته باشد.
- `chunk_count` با تعداد واقعی Chunkهای هر Document برابر باشد.
- `document_id` با `source_id` سازگار باشد.
- Metadata هویتی و Attribution با Processed V2 یکسان باشد.
- `raw_content_hash` هر Chunk با Processed V2 پذیرفته‌شده برابر باشد.
- Hash موجود در Processed V2 نیز قبلاً با Raw HTML واقعی تأیید شده است.

این Validation درباره مناسب بودن اندازه Chunk یا کیفیت روش Chunking تصمیم نمی‌گیرد.

انتخاب و Benchmark روش Chunking در Notebook مخصوص آن انجام خواهد شد.

**معیار PASS:**

تمام Validationهای ساختاری و Lineage بدون هیچ ناسازگاری عبور کنند.

در صورت PASS شدن این مرحله، `website_chunks_v3.json` به‌عنوان Chunk Artifact فنی پذیرفته‌شده فعلی ثبت خواهد شد؛ نه به‌عنوان نسخه نهایی Benchmark‌شده Chunking.

</div>

In [36]:
# Code Cell 30

with chunks_v3_path.open("r", encoding="utf-8") as file:
    accepted_chunks_data = json.load(file)

assert isinstance(accepted_chunks_data, list)

accepted_chunks_by_source_id = defaultdict(list)

for chunk in accepted_chunks_data:
    accepted_chunks_by_source_id[chunk["source_id"]].append(chunk)

accepted_chunk_ids = [
    chunk["chunk_id"]
    for chunk in accepted_chunks_data
]

accepted_chunk_source_ids = set(
    accepted_chunks_by_source_id
)

missing_documents = sorted(
    accepted_processed_source_ids - accepted_chunk_source_ids
)

extra_documents = sorted(
    accepted_chunk_source_ids - accepted_processed_source_ids
)

chunk_index_anomalies = []
chunk_count_mismatches = []
document_source_id_mismatches = []

for source_id, document_chunks in accepted_chunks_by_source_id.items():
    ordered_chunks = sorted(
        document_chunks,
        key=lambda item: item["chunk_index"],
    )

    actual_chunk_count = len(ordered_chunks)

    expected_indexes = list(
        range(1, actual_chunk_count + 1)
    )

    actual_indexes = [
        chunk["chunk_index"]
        for chunk in ordered_chunks
    ]

    declared_counts = {
        chunk["chunk_count"]
        for chunk in ordered_chunks
    }

    if actual_indexes != expected_indexes:
        chunk_index_anomalies.append(source_id)

    if (
        len(declared_counts) != 1
        or next(iter(declared_counts)) != actual_chunk_count
    ):
        chunk_count_mismatches.append(source_id)

    for chunk in ordered_chunks:
        if chunk["document_id"] != chunk["source_id"]:
            document_source_id_mismatches.append(
                chunk["chunk_id"]
            )

lineage_fields = [
    "title",
    "author",
    "content_origin",
    "knowledge_owner",
    "canonical_url",
    "source_type",
    "raw_content_hash",
]

final_chunk_lineage_mismatches = {
    field_name: []
    for field_name in lineage_fields
}

for chunk in accepted_chunks_data:
    source_id = chunk["source_id"]
    processed_record = accepted_processed_by_source_id[source_id]

    for field_name in lineage_fields:
        if chunk[field_name] != processed_record[field_name]:
            final_chunk_lineage_mismatches[field_name].append(
                chunk["chunk_id"]
            )

print("Accepted chunks:", len(accepted_chunks_data))
print("Unique chunk IDs:", len(set(accepted_chunk_ids)))
print("Documents represented:", len(accepted_chunk_source_ids))

print("\nCoverage:")
print("  Missing documents:", len(missing_documents))
print("  Extra documents:", len(extra_documents))

print("\nStructure:")
print("  Chunk index anomalies:", len(chunk_index_anomalies))
print("  Chunk count mismatches:", len(chunk_count_mismatches))
print(
    "  document_id/source_id mismatches:",
    len(document_source_id_mismatches),
)

print("\nLineage mismatches:")

for field_name, mismatches in final_chunk_lineage_mismatches.items():
    print(f"  {field_name}: {len(mismatches)}")

assert len(accepted_chunks_data) == 1116
assert len(set(accepted_chunk_ids)) == 1116
assert len(accepted_chunk_source_ids) == 529
assert len(missing_documents) == 0
assert len(extra_documents) == 0
assert len(chunk_index_anomalies) == 0
assert len(chunk_count_mismatches) == 0
assert len(document_source_id_mismatches) == 0

for mismatches in final_chunk_lineage_mismatches.values():
    assert len(mismatches) == 0

print("\nCHUNKS V3 INDEPENDENT ACCEPTANCE GATE: PASS")

Accepted chunks: 1116
Unique chunk IDs: 1116
Documents represented: 529

Coverage:
  Missing documents: 0
  Extra documents: 0

Structure:
  Chunk index anomalies: 0
  Chunk count mismatches: 0
  document_id/source_id mismatches: 0

Lineage mismatches:
  title: 0
  author: 0
  content_origin: 0
  knowledge_owner: 0
  canonical_url: 0
  source_type: 0
  raw_content_hash: 0

CHUNKS V3 INDEPENDENT ACCEPTANCE GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### اعتبارسنجی محتوای Chunkها نسبت به Document اصلی

نسخه `website_chunks_v3.json` از نظر ساختار، پوشش و Metadata با موفقیت پذیرفته شد.

نتیجه مرحله قبل:

- 1116 Chunk
- 1116 `chunk_id` یکتا
- پوشش کامل هر 529 Document
- Document گم‌شده: صفر
- Document اضافه: صفر
- خطای ترتیب Chunk: صفر
- اختلاف Lineage: صفر

با این حال، برای تکمیل Lineage باید خود متن Chunkها نیز به Document اصلی متصل باشد.

وجود Metadata صحیح کافی نیست؛ ممکن است Metadata یک Chunk به Document درست اشاره کند اما متن داخل Chunk متعلق به آن Document نباشد.

در این مرحله بررسی می‌شود:

- `content` هیچ Chunkی خالی نباشد.
- متن هر Chunk واقعاً در `content` مربوط به Parent Document در Processed V2 وجود داشته باشد.
- `chunk_id` با `source_id` و `chunk_index` همان Chunk سازگار باشد.
- `article_id` موجود در `metadata` با شناسه موجود در `source_id` سازگار باشد.
- هیچ Chunk بدون Parent Document وجود نداشته باشد.

در این مرحله درباره اندازه، Overlap یا کیفیت Semantic Chunking تصمیم‌گیری نمی‌شود.

هدف فقط اثبات این زنجیره است:

`Raw → Interim → Processed V2 → Chunk V3`

**معیار PASS:**

- Chunk خالی: صفر
- Chunk خارج از متن Parent: صفر
- `chunk_id` ناسازگار: صفر
- `article_id` ناسازگار: صفر

</div>

In [37]:
# Code Cell 31

empty_chunk_ids = []
chunk_content_parent_mismatches = []
chunk_id_contract_mismatches = []
chunk_article_id_mismatches = []
missing_parent_documents = []

for chunk in accepted_chunks_data:
    chunk_id = chunk["chunk_id"]
    source_id = chunk["source_id"]
    chunk_index = chunk["chunk_index"]
    chunk_content = chunk["content"]

    if source_id not in accepted_processed_by_source_id:
        missing_parent_documents.append(chunk_id)
        continue

    parent_document = accepted_processed_by_source_id[source_id]
    parent_content = parent_document["content"]

    if not isinstance(chunk_content, str) or not chunk_content.strip():
        empty_chunk_ids.append(chunk_id)

    elif chunk_content.strip() not in parent_content:
        chunk_content_parent_mismatches.append(chunk_id)

    expected_chunk_id = f"{source_id}:chunk:{chunk_index}"

    if chunk_id != expected_chunk_id:
        chunk_id_contract_mismatches.append(
            {
                "chunk_id": chunk_id,
                "expected": expected_chunk_id,
            }
        )

    expected_article_id = source_id.split(":", 1)[1]
    chunk_article_id = chunk["metadata"].get("article_id")

    if chunk_article_id != expected_article_id:
        chunk_article_id_mismatches.append(
            {
                "chunk_id": chunk_id,
                "article_id": chunk_article_id,
                "expected": expected_article_id,
            }
        )

print("Chunks checked:", len(accepted_chunks_data))
print("Missing parent documents:", len(missing_parent_documents))
print("Empty chunks:", len(empty_chunk_ids))
print(
    "Chunk content not found in parent document:",
    len(chunk_content_parent_mismatches),
)
print(
    "Chunk ID contract mismatches:",
    len(chunk_id_contract_mismatches),
)
print(
    "Chunk article_id mismatches:",
    len(chunk_article_id_mismatches),
)

assert len(missing_parent_documents) == 0
assert len(empty_chunk_ids) == 0
assert len(chunk_content_parent_mismatches) == 0
assert len(chunk_id_contract_mismatches) == 0
assert len(chunk_article_id_mismatches) == 0

print("\nCHUNK CONTENT TO PARENT LINEAGE GATE: PASS")

Chunks checked: 1116
Missing parent documents: 0
Empty chunks: 0
Chunk content not found in parent document: 0
Chunk ID contract mismatches: 0
Chunk article_id mismatches: 0

CHUNK CONTENT TO PARENT LINEAGE GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### بررسی آمادگی Citation Locator در Chunkهای Website

ارتباط محتوایی Chunkها با Parent Document با موفقیت تأیید شد.

نتیجه مرحله قبل:

- Chunk بدون Parent: صفر
- Chunk خالی: صفر
- Chunk خارج از متن Parent: صفر
- `chunk_id` ناسازگار: صفر
- `article_id` ناسازگار: صفر

بنابراین زنجیره محتوایی زیر اکنون قابل اثبات است:

`Raw → Interim → Processed V2 → Chunk V3`

مرحله بعد مربوط به **Citation Readiness** است.

برای یک سیستم RAG قابل استناد، صرف داشتن URL مقاله کافی نیست. هنگام تولید پاسخ باید بتوانیم مشخص کنیم یک Claim از کدام بخش مشخص Source آمده است.

برای Website حداقل اجزای قابل استفاده برای Citation عبارت‌اند از:

- `article_id`
- `canonical_url`
- `chunk_id`
- `chunk_index`
- محل دقیق متن Chunk در Parent Document

Chunk Artifact فعلی دارای شناسه مقاله، URL و Chunk ID است، اما در Schema فعلی فیلد صریحی مانند:

- `char_start`
- `char_end`
- `paragraph_start`
- `paragraph_end`

مشاهده نشده است.

در این مرحله هنوز Citation Locator جدیدی ساخته نمی‌شود.

ابتدا بررسی می‌کنیم آیا موقعیت دقیق هر Chunk را می‌توان به‌صورت Deterministic از متن Parent Document استخراج کرد یا خیر.

برای هر Chunk:

1. متن Chunk در Parent Document جستجو می‌شود.
2. تعداد دفعات وقوع متن Chunk در Parent اندازه‌گیری می‌شود.
3. اگر دقیقاً یک بار وجود داشته باشد، `char_start` و `char_end` به‌صورت قابل اعتماد قابل استخراج هستند.
4. اگر متن چند بار در Parent تکرار شده باشد، Locator مبتنی بر جستجوی ساده Ambiguous خواهد بود.
5. اگر متن پیدا نشود، Citation Lineage آن Chunk نامعتبر خواهد بود.

این مرحله فقط Citation Readiness را اندازه‌گیری می‌کند و Artifact جدیدی ایجاد نمی‌کند.

**معیار PASS برای Readiness:**

- Chunk پیدا نشده: صفر
- تمام Chunkها حداقل یک Match در Parent داشته باشند.

تعداد Chunkهای دارای Match چندگانه نیز جداگانه گزارش می‌شود تا مشخص شود برای ساخت Locator نهایی به منطق دقیق‌تری نیاز داریم یا خیر.

</div>

In [38]:
# Code Cell 32

citation_locator_diagnostics = []

not_found_chunk_ids = []
unique_match_chunk_ids = []
multiple_match_chunk_ids = []

for chunk in accepted_chunks_data:
    chunk_id = chunk["chunk_id"]
    source_id = chunk["source_id"]
    chunk_content = chunk["content"]

    parent_content = accepted_processed_by_source_id[source_id]["content"]

    match_positions = []
    search_start = 0

    while True:
        position = parent_content.find(
            chunk_content,
            search_start,
        )

        if position == -1:
            break

        match_positions.append(position)
        search_start = position + 1

    match_count = len(match_positions)

    if match_count == 0:
        not_found_chunk_ids.append(chunk_id)
        char_start = None
        char_end = None

    elif match_count == 1:
        unique_match_chunk_ids.append(chunk_id)
        char_start = match_positions[0]
        char_end = char_start + len(chunk_content)

    else:
        multiple_match_chunk_ids.append(chunk_id)
        char_start = None
        char_end = None

    citation_locator_diagnostics.append(
        {
            "chunk_id": chunk_id,
            "source_id": source_id,
            "article_id": chunk["metadata"]["article_id"],
            "canonical_url": chunk["canonical_url"],
            "chunk_index": chunk["chunk_index"],
            "match_count": match_count,
            "char_start": char_start,
            "char_end": char_end,
        }
    )

print("Chunks evaluated:", len(citation_locator_diagnostics))

print("\nCitation locator readiness:")
print("  Not found in parent:", len(not_found_chunk_ids))
print("  Unique text match:", len(unique_match_chunk_ids))
print("  Multiple text matches:", len(multiple_match_chunk_ids))

if multiple_match_chunk_ids:
    print("\nSample ambiguous chunks:")
    for chunk_id in multiple_match_chunk_ids[:10]:
        diagnostic = next(
            item
            for item in citation_locator_diagnostics
            if item["chunk_id"] == chunk_id
        )

        print(
            f"  {chunk_id} | "
            f"matches={diagnostic['match_count']}"
        )

assert len(citation_locator_diagnostics) == 1116
assert len(not_found_chunk_ids) == 0

print("\nWEBSITE CITATION LOCATOR READINESS GATE: PASS")

Chunks evaluated: 1116

Citation locator readiness:
  Not found in parent: 0
  Unique text match: 1116
  Multiple text matches: 0

WEBSITE CITATION LOCATOR READINESS GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### ساخت Citation Locator قطعی برای تمام Chunkهای Website

بررسی Citation Readiness نشان داد هر 1116 Chunk دقیقاً یک بار در Parent Document مربوط به خود پیدا می‌شود.

نتیجه:

- Chunk پیدا نشده در Parent: صفر
- Chunk دارای Match یکتا: 1116
- Chunk دارای Match چندگانه: صفر

بنابراین برای تمام Chunkهای فعلی Website می‌توان بدون حدس و به‌صورت Deterministic موقعیت دقیق متن را در Document اصلی ثبت کرد.

برای هر Chunk دو Locator اصلی ساخته می‌شود:

- `char_start`: موقعیت شروع متن Chunk در Parent Document
- `char_end`: موقعیت پایان متن Chunk در Parent Document

این Locator در کنار موارد زیر ذخیره خواهد شد:

- `chunk_id`
- `source_id`
- `article_id`
- `canonical_url`
- `chunk_index`

همچنین SHA-256 خود متن Chunk ثبت می‌شود تا بعداً بتوان بررسی کرد Locator به همان متن مورد انتظار اشاره می‌کند.

Artifact جدید:

`data/phase2/processed/website_citation_locators_v1.json`

این فایل Chunkها را تغییر نمی‌دهد و فقط لایه Citation Locator را به‌صورت مستقل نگهداری می‌کند.

پس از ساخت Artifact دوباره بررسی می‌شود که:

- برای هر 1116 Chunk دقیقاً یک Locator وجود داشته باشد.
- تمام `chunk_id`ها Unique باشند.
- هیچ Locator خارج از محدوده Parent Document نباشد.
- متن Parent در بازه `char_start:char_end` دقیقاً با متن Chunk برابر باشد.
- Hash متن Locator با Hash واقعی Chunk برابر باشد.

**معیار PASS:**

تمام 1116 Locator باید بدون هیچ ناسازگاری قابل بازسازی و اعتبارسنجی باشند.

</div>

In [39]:
# Code Cell 33

citation_locator_path = (
    project_root
    / "data"
    / "phase2"
    / "processed"
    / "website_citation_locators_v1.json"
)

citation_locators = []

for chunk in accepted_chunks_data:
    chunk_id = chunk["chunk_id"]
    source_id = chunk["source_id"]
    chunk_content = chunk["content"]

    parent_content = accepted_processed_by_source_id[source_id]["content"]

    char_start = parent_content.find(chunk_content)

    assert char_start >= 0

    char_end = char_start + len(chunk_content)

    chunk_text_hash = hashlib.sha256(
        chunk_content.encode("utf-8")
    ).hexdigest()

    citation_locators.append(
        {
            "chunk_id": chunk_id,
            "source_id": source_id,
            "article_id": chunk["metadata"]["article_id"],
            "canonical_url": chunk["canonical_url"],
            "chunk_index": chunk["chunk_index"],
            "char_start": char_start,
            "char_end": char_end,
            "chunk_text_sha256": chunk_text_hash,
        }
    )

citation_locator_path.write_text(
    json.dumps(
        citation_locators,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

with citation_locator_path.open("r", encoding="utf-8") as file:
    validated_citation_locators = json.load(file)

locator_chunk_ids = [
    locator["chunk_id"]
    for locator in validated_citation_locators
]

locator_validation_errors = []

accepted_chunks_by_id = {
    chunk["chunk_id"]: chunk
    for chunk in accepted_chunks_data
}

for locator in validated_citation_locators:
    chunk_id = locator["chunk_id"]
    source_id = locator["source_id"]

    chunk = accepted_chunks_by_id[chunk_id]
    parent_content = accepted_processed_by_source_id[source_id]["content"]

    char_start = locator["char_start"]
    char_end = locator["char_end"]

    located_text = parent_content[
        char_start:char_end
    ]

    located_text_hash = hashlib.sha256(
        located_text.encode("utf-8")
    ).hexdigest()

    if located_text != chunk["content"]:
        locator_validation_errors.append(
            {
                "chunk_id": chunk_id,
                "error": "text_mismatch",
            }
        )

    if located_text_hash != locator["chunk_text_sha256"]:
        locator_validation_errors.append(
            {
                "chunk_id": chunk_id,
                "error": "hash_mismatch",
            }
        )

    if not (
        0 <= char_start < char_end <= len(parent_content)
    ):
        locator_validation_errors.append(
            {
                "chunk_id": chunk_id,
                "error": "invalid_range",
            }
        )

print(
    "Citation locator artifact:",
    citation_locator_path.relative_to(project_root),
)

print("Locators:", len(validated_citation_locators))
print("Unique chunk IDs:", len(set(locator_chunk_ids)))
print("Validation errors:", len(locator_validation_errors))

assert len(validated_citation_locators) == 1116
assert len(set(locator_chunk_ids)) == 1116
assert set(locator_chunk_ids) == set(accepted_chunks_by_id)
assert len(locator_validation_errors) == 0

print("\nWEBSITE CITATION LOCATOR ARTIFACT GATE: PASS")

Citation locator artifact: data/phase2/processed/website_citation_locators_v1.json
Locators: 1116
Unique chunk IDs: 1116
Validation errors: 0

WEBSITE CITATION LOCATOR ARTIFACT GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### ساخت گزارش نهایی Website Acceptance

Citation Locator برای تمام 1116 Chunk با موفقیت ساخته و اعتبارسنجی شد.

نتیجه:

- Locator ساخته‌شده: 1116
- `chunk_id` یکتا: 1116
- Locator نامعتبر: صفر
- متن بازیابی‌شده از Parent با متن Chunk ناسازگار: صفر

بنابراین در وضعیت فعلی می‌توان هر Chunk را به بخش دقیق متن Parent Document متصل کرد.

اکنون نتایج اصلی Notebook 02 در یک Acceptance Report ساختاریافته ثبت می‌شوند.

این گزارش باید مشخص کند کدام Artifactها:

- ورودی اولیه بوده‌اند.
- مشکل داشته‌اند.
- Superseded شده‌اند.
- اصلاح شده‌اند.
- برای ادامه فاز دوم پذیرفته شده‌اند.

### وضعیت Artifactهای Website

Manifest اصلی:

`data/phase2/manifests/website_manifest_v1.json`

وضعیت:

`ACCEPTED AS SOURCE MANIFEST`

تعداد رکورد:

529

---

Interim Artifact:

`data/phase2/interim/website_extracted_v2.json`

وضعیت:

`ACCEPTED`

تعداد رکورد:

529

---

Processed V1:

`data/phase2/processed/website_documents_v1.json`

وضعیت:

`SUPERSEDED`

علت:

وجود 3 مقدار ناسازگار `raw_content_hash`

---

Processed V2:

`data/phase2/processed/website_documents_v2.json`

وضعیت:

`ACCEPTED`

تعداد Document:

529

---

Chunks V2:

`data/phase2/chunks/website_chunks_v2.json`

وضعیت:

`SUPERSEDED`

علت:

وجود 5 Chunk دارای `raw_content_hash` قدیمی

---

Chunks V3:

`data/phase2/chunks/website_chunks_v3.json`

وضعیت:

`ACCEPTED FOR LINEAGE`

تعداد Chunk:

1116

این Acceptance فقط ساختار، هویت، Content Lineage و Metadata را تأیید می‌کند.

روش Chunking فعلی هنوز Benchmark نشده است و پذیرش نهایی استراتژی Chunking در Notebook مخصوص Chunking Benchmark انجام خواهد شد.

---

Citation Locator:

`data/phase2/processed/website_citation_locators_v1.json`

وضعیت:

`ACCEPTED`

تعداد Locator:

1116

### وضعیت Attribution

Website Corpus شامل:

- 76 مقاله `primary_author`
- 453 مقاله `supervised`

است.

تمام 529 رکورد دارای:

`knowledge_owner = Dr. Komeil Roudi`

هستند.

این تفکیک باید در مراحل بعدی Retrieval و Generation حفظ شود.

### وضعیت Content Quality

تمام 529 رکورد حفظ شده‌اند.

Quality Flagهای شناسایی‌شده شامل:

- 3 متن کمتر از 300 کاراکتر
- 13 متن کمتر از 500 کاراکتر
- 29 رکورد عضو گروه‌های دارای عنوان تکراری
- 2 رکورد دارای متن دقیقاً یکسان
- 24 رکورد دارای نشانه چندرسانه‌ای در عنوان

این موارد به‌تنهایی باعث حذف Source نشده‌اند.

اکنون این نتایج در یک Acceptance Report نسخه‌دار ذخیره می‌شوند:

`reports/phase2/website_acceptance_v1.json`

**معیار PASS:**

گزارش باید با موفقیت ساخته شود و تمام Counts و وضعیت Artifactها مطابق نتایج واقعی همین Notebook باشند.

</div>

In [40]:
# Code Cell 34

website_acceptance_report_path = (
    project_root
    / "reports"
    / "phase2"
    / "website_acceptance_v1.json"
)

website_acceptance_report = {
    "schema_version": "1.0",
    "notebook": "02_website_acceptance.ipynb",
    "source": {
        "source_type": "website",
        "knowledge_owner": "Dr. Komeil Roudi",
        "manifest": {
            "path": "data/phase2/manifests/website_manifest_v1.json",
            "sha256": manifest_sha256,
            "record_count": 529,
            "status": "accepted",
        },
    },
    "attribution": {
        "primary_author": 76,
        "supervised": 453,
        "anomalies": 0,
    },
    "raw": {
        "accepted_article_count": 529,
        "missing_raw": 0,
        "missing_html": 0,
        "missing_source_json": 0,
        "hash_mismatches": 0,
        "extra_non_manifest_artifacts": [
            "TEST_INVALID"
        ],
    },
    "interim": {
        "path": "data/phase2/interim/website_extracted_v2.json",
        "record_count": 529,
        "status": "accepted",
    },
    "processed": {
        "superseded": {
            "path": "data/phase2/processed/website_documents_v1.json",
            "reason": "3 stale raw_content_hash values",
        },
        "accepted": {
            "path": "data/phase2/processed/website_documents_v2.json",
            "document_count": 529,
            "remaining_lineage_mismatches": 0,
            "status": "accepted",
        },
    },
    "chunks": {
        "superseded": {
            "path": "data/phase2/chunks/website_chunks_v2.json",
            "reason": "5 stale raw_content_hash values across 3 documents",
        },
        "accepted_for_lineage": {
            "path": "data/phase2/chunks/website_chunks_v3.json",
            "chunk_count": 1116,
            "document_count": 529,
            "chunk_id_duplicates": 0,
            "missing_documents": 0,
            "extra_documents": 0,
            "lineage_mismatches": 0,
            "status": "accepted_for_lineage_not_chunking_benchmark",
        },
    },
    "citation": {
        "path": "data/phase2/processed/website_citation_locators_v1.json",
        "locator_count": 1116,
        "unique_chunk_ids": 1116,
        "validation_errors": 0,
        "status": "accepted",
    },
    "quality_flags": {
        "records_retained": 529,
        "short_under_300": 3,
        "short_under_500": 13,
        "duplicate_title_members": 29,
        "exact_body_duplicate_members": 2,
        "media_title_indicator": 24,
    },
    "acceptance_status": "PASS",
    "limitations": [
        "Chunking strategy has not been benchmarked yet.",
        "Short multimedia-related website text is retained as valid source content.",
        "Exact duplicate body text is flagged but source records are retained.",
        "Rights evidence is not established by this notebook.",
    ],
}

website_acceptance_report_path.write_text(
    json.dumps(
        website_acceptance_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

with website_acceptance_report_path.open(
    "r",
    encoding="utf-8",
) as file:
    validated_acceptance_report = json.load(file)

print(
    "Acceptance report:",
    website_acceptance_report_path.relative_to(project_root),
)

print(
    "Acceptance status:",
    validated_acceptance_report["acceptance_status"],
)

print(
    "Website records:",
    validated_acceptance_report["source"]["manifest"]["record_count"],
)

print(
    "Accepted processed documents:",
    validated_acceptance_report["processed"]["accepted"]["document_count"],
)

print(
    "Accepted lineage chunks:",
    validated_acceptance_report["chunks"]["accepted_for_lineage"]["chunk_count"],
)

print(
    "Citation locators:",
    validated_acceptance_report["citation"]["locator_count"],
)

assert validated_acceptance_report["acceptance_status"] == "PASS"
assert validated_acceptance_report["source"]["manifest"]["record_count"] == 529
assert validated_acceptance_report["attribution"]["primary_author"] == 76
assert validated_acceptance_report["attribution"]["supervised"] == 453
assert validated_acceptance_report["raw"]["hash_mismatches"] == 0
assert validated_acceptance_report["processed"]["accepted"]["document_count"] == 529
assert validated_acceptance_report["chunks"]["accepted_for_lineage"]["chunk_count"] == 1116
assert validated_acceptance_report["citation"]["locator_count"] == 1116

print("\nWEBSITE ACCEPTANCE REPORT GATE: PASS")

Acceptance report: reports/phase2/website_acceptance_v1.json
Acceptance status: PASS
Website records: 529
Accepted processed documents: 529
Accepted lineage chunks: 1116
Citation locators: 1116

WEBSITE ACCEPTANCE REPORT GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### اعتبارسنجی نهایی Website Acceptance

گزارش `website_acceptance_v1.json` با موفقیت ساخته شد و وضعیت کلی Website Acceptance را `PASS` ثبت کرد.

اکنون آخرین Gate این Notebook اجرا می‌شود.

هدف این مرحله این است که به‌جای اتکا به نتیجه یک Cell خاص، مهم‌ترین Artifactهای پذیرفته‌شده و Contractهای اصلی Website را یک‌جا بررسی کنیم.

Artifactهای نهایی مورد قبول این Notebook عبارت‌اند از:

- Manifest:
  `data/phase2/manifests/website_manifest_v1.json`

- Interim:
  `data/phase2/interim/website_extracted_v2.json`

- Processed پذیرفته‌شده:
  `data/phase2/processed/website_documents_v2.json`

- Chunk Artifact پذیرفته‌شده از نظر Structure و Lineage:
  `data/phase2/chunks/website_chunks_v3.json`

- Citation Locator:
  `data/phase2/processed/website_citation_locators_v1.json`

- Acceptance Report:
  `reports/phase2/website_acceptance_v1.json`

نسخه‌های زیر به‌عنوان Artifact تاریخی حفظ می‌شوند اما نباید ورودی فعال مراحل بعدی باشند:

- `website_documents_v1.json`
- `website_chunks_v2.json`

### موارد Final Validation

در Gate نهایی بررسی می‌شود:

- Manifest دارای 529 رکورد باشد.
- Attribution شامل 76 رکورد `primary_author` و 453 رکورد `supervised` باشد.
- هر 529 رکورد Raw معتبر و دارای Integrity تأییدشده باشند.
- Processed V2 دارای 529 Document یکتا باشد.
- Chunk V3 دارای 1116 Chunk یکتا و پوشش کامل 529 Document باشد.
- Citation Locator برای تمام 1116 Chunk وجود داشته باشد.
- هیچ Lineage mismatch در Artifactهای پذیرفته‌شده باقی نمانده باشد.
- Acceptance Report وضعیت `PASS` داشته باشد.
- Artifactهای فعال دقیقاً نسخه‌های اصلاح‌شده باشند.

### نکته درباره Failureهای میانی

در طول این Notebook دو Failure واقعی مشاهده شد:

- سه `raw_content_hash` ناسازگار در Processed V1
- پنج Chunk دارای همان Metadata قدیمی در Chunks V2

این Failureها حذف یا مخفی نشدند.

Root Cause آن‌ها ثبت شد و نسخه‌های جدید با حفظ نسخه‌های تاریخی ساخته شدند.

همچنین یک False Positive در Validation مربوط به `chunk_index` شناسایی شد که ناشی از فرض اشتباه 0-based indexing بود؛ Contract واقعی Pipeline برابر 1-based indexing است.

بنابراین وجود گزارش `FAIL` در مراحل اکتشافی این Notebook به معنی Fail بودن نتیجه نهایی نیست.

معیار نهایی این است که Artifactهای پذیرفته‌شده فعلی بدون ناسازگاری باشند.

**معیار PASS نهایی:**

تمام Assertionهای این Cell بدون خطا عبور کنند و خروجی نهایی برابر باشد با:

`NOTEBOOK 02 FINAL ACCEPTANCE: PASS`

</div>

In [41]:
# Code Cell 35

final_manifest_count = len(website_manifest)

final_attribution_distribution = Counter(
    record["content_origin"]
    for record in website_manifest
)

final_processed_source_ids = {
    record["source_id"]
    for record in accepted_processed_data
}

final_processed_document_ids = {
    record["document_id"]
    for record in accepted_processed_data
}

final_chunk_ids = {
    chunk["chunk_id"]
    for chunk in accepted_chunks_data
}

final_chunk_source_ids = {
    chunk["source_id"]
    for chunk in accepted_chunks_data
}

final_locator_chunk_ids = {
    locator["chunk_id"]
    for locator in validated_citation_locators
}

assert final_manifest_count == 529

assert final_attribution_distribution == Counter(
    {
        "supervised": 453,
        "primary_author": 76,
    }
)

assert len(hash_mismatches) == 0
assert len(size_mismatches) == 0
assert len(article_id_mismatches) == 0
assert len(raw_url_mismatches) == 0
assert len(raw_canonical_url_mismatches) == 0

assert len(accepted_processed_data) == 529
assert len(final_processed_source_ids) == 529
assert len(final_processed_document_ids) == 529

assert len(accepted_chunks_data) == 1116
assert len(final_chunk_ids) == 1116
assert len(final_chunk_source_ids) == 529

assert final_chunk_source_ids == final_processed_source_ids

assert len(validated_citation_locators) == 1116
assert len(final_locator_chunk_ids) == 1116
assert final_locator_chunk_ids == final_chunk_ids

for mismatches in validation_mismatches.values():
    assert len(mismatches) == 0

for mismatches in final_chunk_lineage_mismatches.values():
    assert len(mismatches) == 0

assert len(locator_validation_errors) == 0

assert (
    validated_acceptance_report["acceptance_status"]
    == "PASS"
)

assert (
    validated_acceptance_report["processed"]["accepted"]["path"]
    == "data/phase2/processed/website_documents_v2.json"
)

assert (
    validated_acceptance_report["chunks"]["accepted_for_lineage"]["path"]
    == "data/phase2/chunks/website_chunks_v3.json"
)

print("Manifest records:", final_manifest_count)
print(
    "Primary-author records:",
    final_attribution_distribution["primary_author"],
)
print(
    "Supervised records:",
    final_attribution_distribution["supervised"],
)
print("Accepted processed documents:", len(final_processed_source_ids))
print("Accepted chunks:", len(final_chunk_ids))
print("Citation locators:", len(final_locator_chunk_ids))
print("Remaining processed lineage mismatches: 0")
print("Remaining chunk lineage mismatches: 0")
print("Citation locator errors: 0")

print("\nNOTEBOOK 02 FINAL ACCEPTANCE: PASS")

Manifest records: 529
Primary-author records: 76
Supervised records: 453
Accepted processed documents: 529
Accepted chunks: 1116
Citation locators: 1116
Remaining processed lineage mismatches: 0
Remaining chunk lineage mismatches: 0
Citation locator errors: 0

NOTEBOOK 02 FINAL ACCEPTANCE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

## جمع‌بندی نهایی Notebook 02 — Website Acceptance

Notebook `02_website_acceptance.ipynb` با هدف بررسی، اعتبارسنجی و پذیرش خروجی Website Ingestion اجرا شد.

در این Notebook هیچ Crawl مجددی انجام نشد و Dataset موجود حاصل از `01_website_ingestion.ipynb` مورد Audit و Acceptance قرار گرفت.

### Website Corpus نهایی

Manifest اصلی شامل 529 رکورد معتبر است.

تفکیک Attribution:

- 76 مقاله `primary_author`
- 453 مقاله `supervised`

در تمام 529 رکورد:

`knowledge_owner = Dr. Komeil Roudi`

است.

این تفکیک باید در مراحل بعدی Retrieval و Generation حفظ شود تا محتوای تحت نظارت با نوشته مستقیم دکتر کمیل رودی اشتباه گرفته نشود.

### Raw Data

برای هر 529 مقاله:

- `article.html` موجود است.
- `source.json` موجود است.
- HTTP Status برابر 200 است.
- `is_valid_article = True`
- `has_h1 = True`
- `has_content_container = True`
- Hash فایل Raw با `source.json` برابر است.
- اندازه فایل با Metadata ذخیره‌شده برابر است.
- URL و Canonical URL با Manifest سازگار هستند.

Missing Raw:

`0`

Raw Hash Mismatch:

`0`

یک Artifact خارج از Corpus با شناسه `TEST_INVALID` شناسایی شد که در Manifest حضور ندارد و در Metadata آن `is_valid_article = false` ثبت شده است؛ بنابراین وارد Dataset پذیرفته‌شده نمی‌شود.

### Interim Artifact

Artifact زیر شامل 529 رکورد است:

`data/phase2/interim/website_extracted_v2.json`

تمام 529 رکورد دارای Title و Body Text غیرخالی هستند.

Lineage میان Manifest، Raw و Interim بدون ناسازگاری تأیید شد.

### Content Quality

تمام 529 رکورد حفظ شدند.

Quality Flagهای شناسایی‌شده:

- متن کمتر از 300 کاراکتر: 3
- متن کمتر از 500 کاراکتر: 13
- رکوردهای عضو گروه‌های دارای عنوان تکراری: 29
- رکوردهای دارای متن دقیقاً یکسان: 2
- رکوردهای دارای نشانه چندرسانه‌ای در عنوان: 24

کوتاه بودن متن به‌تنهایی معیار حذف Source در نظر گرفته نشد.

همچنین Duplicate متنی باعث حذف خودکار Source نشد و فقط برای Deduplication و Retrieval آینده Flag شد.

### Processed Artifact

در Artifact اولیه:

`data/phase2/processed/website_documents_v1.json`

سه مقدار قدیمی یا ناسازگار `raw_content_hash` شناسایی شد.

Root Cause نشان داد:

- Raw HTML صحیح بود.
- `source.json` صحیح بود.
- Interim صحیح بود.
- Content پردازش‌شده صحیح بود.
- فقط Metadata مربوط به Hash در Processed V1 قدیمی بود.

نسخه تاریخی V1 حفظ شد و نسخه اصلاح‌شده زیر ساخته شد:

`data/phase2/processed/website_documents_v2.json`

Processed V2 دارای:

- 529 Document
- 529 `source_id` یکتا
- 529 `document_id` یکتا
- صفر Content Mismatch
- صفر Attribution Mismatch
- صفر Raw Hash Mismatch

است.

بنابراین `website_documents_v2.json` Processed Artifact پذیرفته‌شده Website است.

### Chunk Artifact

Artifact قبلی:

`data/phase2/chunks/website_chunks_v2.json`

دارای 1116 Chunk بود.

بررسی مشخص کرد Chunk Structure سالم است و Pipeline از 1-based indexing استفاده می‌کند.

پنج Chunk دارای Hash قدیمی بودند که دقیقاً متعلق به همان سه Document دارای مشکل در Processed V1 بودند.

هیچ Rechunking انجام نشد.

نسخه اصلاح‌شده زیر ساخته شد:

`data/phase2/chunks/website_chunks_v3.json`

نتیجه Validation:

- Chunkها: 1116
- `chunk_id` یکتا: 1116
- Documentهای پوشش‌داده‌شده: 529
- Missing Document: صفر
- Extra Document: صفر
- Chunk Index Anomaly: صفر
- Chunk Count Mismatch: صفر
- Content خارج از Parent Document: صفر
- Lineage Mismatch: صفر

این Artifact از نظر Structure و Lineage پذیرفته شده است.

استراتژی Chunking فعلی هنوز Benchmark نشده و پذیرش نهایی روش Chunking در Notebook مخصوص Chunking Benchmark انجام خواهد شد.

### Citation Locator

برای هر 1116 Chunk مشخص شد متن آن دقیقاً یک بار در Parent Document ظاهر می‌شود.

بنابراین Locator قطعی مبتنی بر Character Offset برای تمام Chunkها ساخته شد:

`data/phase2/processed/website_citation_locators_v1.json`

نتیجه:

- Locator: 1116
- Unique Chunk ID: 1116
- Invalid Locator: صفر
- Text Mismatch: صفر

بنابراین اتصال:

`Chunk → Document → Website Source`

در سطح فعلی قابل ردیابی است.

### Artifactهای پذیرفته‌شده

Manifest:

`data/phase2/manifests/website_manifest_v1.json`

Interim:

`data/phase2/interim/website_extracted_v2.json`

Processed:

`data/phase2/processed/website_documents_v2.json`

Chunks از نظر Structure و Lineage:

`data/phase2/chunks/website_chunks_v3.json`

Citation Locator:

`data/phase2/processed/website_citation_locators_v1.json`

Acceptance Report:

`reports/phase2/website_acceptance_v1.json`

### Artifactهای Superseded

Processed V1:

`data/phase2/processed/website_documents_v1.json`

Chunks V2:

`data/phase2/chunks/website_chunks_v2.json`

این Artifactها برای Audit Trail حفظ می‌شوند اما نباید ورودی فعال مراحل جدید باشند.

### محدودیت‌های باقی‌مانده

این Notebook موارد زیر را نهایی نکرده است:

- Benchmark استراتژی Chunking
- Deduplication نهایی Cross-source
- Rights Evidence
- Retrieval Benchmark
- Embedding Benchmark
- Reranking
- Generation
- Claim-level Citation Evaluation
- Safety و Answerability

هر کدام در Notebook مخصوص خود بررسی خواهند شد.

### نتیجه نهایی

تمام Gateهای نهایی Website Acceptance با موفقیت عبور کردند.

**NOTEBOOK 02 FINAL ACCEPTANCE: PASS**

### Final Execution Gate

برای بسته‌شدن رسمی Notebook باید اجرای کامل زیر انجام شود:

`Restart Kernel → Run All Cells`

شرط نهایی:

- Zero Error
- تمام Code Cellها به‌ترتیب اجرا شوند.
- آخرین Validation همچنان برابر باشد با:

`NOTEBOOK 02 FINAL ACCEPTANCE: PASS`

</div>